# NEMA-Q — Colab Notebook

**NEMA-Q: Non-Euclidean Memory-Augmented Quantum Graph Neural Network** — instrumentation-first hybrid architecture study.

This notebook recreates the full research codebase on the Colab VM, runs the test suite, and executes demo experiments:

1. Install dependencies
2. Write the `nemaq` package (`%%writefile` cells — edit any cell and re-run to modify the code)
3. Run unit tests (Gates 1–3 checks)
4. Compute Gromov δ-hyperbolicity of Cora (H1 stratification instrument)
5. Train GCN baseline (Gate 2 sanity: ~81% on Cora public split)
6. Train full NEMA-Q (hyperbolic + PQC + bypass) with gradient telemetry (H3)
7. Dequantization control NEMA-C (surrogate) — the headline comparison
8. Attribution demo (H4) + circuit expressibility diagnostics

Runtime: CPU works; GPU (T4) speeds up the classical parts. The PQC is simulated on CPU either way.


## 1. Install dependencies

In [1]:
# Colab ships torch preinstalled; add the rest.
%pip install -q torch_geometric geoopt pennylane pyyaml networkx scipy pytest

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 70.2 MB/s eta 0:00:00
torch 2.11.0+cu128 | cuda: True


## 2. Package source
Each cell writes one module to `/content/src/nemaq/...`.

In [2]:
import pathlib
for d in ["src/nemaq/utils", "src/nemaq/data", "src/nemaq/models",
          "src/nemaq/telemetry", "src/nemaq/training", "src/nemaq/analysis",
          "configs/ablations", "tests", "experiments/runs"]:
    pathlib.Path("/content", d).mkdir(parents=True, exist_ok=True)
for pkg in ["", "utils", "data", "models", "telemetry", "training", "analysis"]:
    (pathlib.Path("/content/src/nemaq") / pkg / "__init__.py").touch()
print("directories ready")


directories ready


In [3]:
%%writefile /content/src/nemaq/utils/seed.py
"""Global reproducibility control.

Every entry point must call `set_seed` before touching any RNG.
Determinism flags are enabled by default; disable only for profiling runs
(and record that in the run manifest).
"""
import os
import random

import numpy as np
import torch


def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # cuBLAS determinism (required by torch.use_deterministic_algorithms on CUDA)
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:
            pass


Writing /content/src/nemaq/utils/seed.py


In [4]:
%%writefile /content/src/nemaq/utils/config.py
"""YAML config loading with base-config inheritance.

A config may declare `inherit: <relative path>`; the child is deep-merged
over the parent. Keeps ablations as small override files (design rule #2:
controls are config swaps, not code forks).
"""
from pathlib import Path

import yaml


def _deep_merge(base: dict, override: dict) -> dict:
    out = dict(base)
    for k, v in override.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge(out[k], v)
        else:
            out[k] = v
    return out


def load_config(path: str | Path) -> dict:
    path = Path(path)
    with open(path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    parent_rel = cfg.pop("inherit", None)
    if parent_rel:
        parent = load_config(path.parent / parent_rel)
        cfg = _deep_merge(parent, cfg)
    return cfg


Writing /content/src/nemaq/utils/config.py


In [5]:
%%writefile /content/src/nemaq/utils/manifest.py
"""Run manifests — the only source of truth for analysis.

Every training run writes exactly one manifest JSON containing the resolved
config, seed, git hash, environment versions, and final metrics. The analysis
layer (`nemaq.analysis`) reads manifests from run directories; numbers are
never copied by hand.
"""
import json
import platform
import subprocess
import time
from pathlib import Path


def _git_hash() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True
        ).strip()
    except Exception:
        return "no-git"


def _env_info() -> dict:
    info = {"python": platform.python_version()}
    for mod in ("torch", "torch_geometric", "geoopt", "pennylane", "numpy"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            info[mod] = "not-installed"
    return info


def write_manifest(run_dir: str | Path, config: dict, seed: int,
                   metrics: dict, telemetry_summary: dict | None = None) -> Path:
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    manifest = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "git_hash": _git_hash(),
        "seed": seed,
        "config": config,
        "env": _env_info(),
        "metrics": metrics,
        "telemetry_summary": telemetry_summary or {},
    }
    out = run_dir / "manifest.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    return out


def read_manifests(root: str | Path) -> list[dict]:
    return [
        json.loads(p.read_text(encoding="utf-8"))
        for p in sorted(Path(root).rglob("manifest.json"))
    ]


Writing /content/src/nemaq/utils/manifest.py


In [6]:
%%writefile /content/src/nemaq/data/loader.py
"""Dataset loading with controlled split protocols.

Split modes:
  - "public":   the standard Planetoid split (comparability with literature).
  - "low_label": k labels per class, seeded — the low-label regime for H2.
  - "ratio":    seeded stratified ratio split (default 30/10/60, HGCN
                protocol) — for datasets without a public split (Disease).
Both return masks on the Data object; the split seed is independent of the
model seed so the same splits are reused across models (paired statistics).
"""
import urllib.request
from pathlib import Path

import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.datasets import Planetoid, Amazon, WebKB
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.utils import to_undirected

PLANETOID = {"cora": "Cora", "citeseer": "CiteSeer", "pubmed": "PubMed"}
AMAZON = {"photo": "Photo", "computers": "Computers"}
WEBKB = {"cornell": "Cornell", "texas": "Texas", "wisconsin": "Wisconsin"}

# Disease (Chami et al. 2019, HGCN repo): synthetic SIR propagation tree,
# delta ~= 0 — the positive control for H1. No public split; use split
# mode "ratio" (HGCN protocol: 30/10/60).
_DISEASE_BASE = ("https://raw.githubusercontent.com/HazyResearch/hgcn/"
                 "master/data/disease_nc/")
_DISEASE_FILES = ("disease_nc.edges.csv", "disease_nc.feats.npz",
                  "disease_nc.labels.npy")


class _InMemoryDataset(list):
    """Minimal ds wrapper so callers can keep using ds[0]."""

    def __init__(self, data: Data):
        super().__init__([data])
        self.num_features = data.num_features
        self.num_classes = int(data.y.max()) + 1


def _load_disease(root: str) -> _InMemoryDataset:
    import scipy.sparse as sp

    ddir = Path(root) / "disease_nc"
    ddir.mkdir(parents=True, exist_ok=True)
    for fname in _DISEASE_FILES:
        dest = ddir / fname
        if not dest.exists():
            urllib.request.urlretrieve(_DISEASE_BASE + fname, dest)

    edges = []
    with open(ddir / "disease_nc.edges.csv", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) != 2 or not parts[0].isdigit():
                continue  # header or malformed line
            edges.append((int(parts[0]), int(parts[1])))
    edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())

    feats = sp.load_npz(ddir / "disease_nc.feats.npz").todense()
    x = torch.tensor(np.asarray(feats), dtype=torch.float)
    x = x / x.sum(dim=-1, keepdim=True).clamp(min=1)  # row-normalize
    y = torch.tensor(np.load(ddir / "disease_nc.labels.npy"),
                     dtype=torch.long).view(-1)
    return _InMemoryDataset(Data(x=x, edge_index=edge_index, y=y))


def load_dataset(name: str, root: str = "data"):
    name = name.lower()
    if name in PLANETOID:
        ds = Planetoid(root, PLANETOID[name], transform=NormalizeFeatures())
    elif name in AMAZON:
        ds = Amazon(root, AMAZON[name], transform=NormalizeFeatures())
    elif name in WEBKB:
        ds = WebKB(root, WEBKB[name], transform=NormalizeFeatures())
    elif name == "disease":
        ds = _load_disease(root)
    else:
        raise ValueError(f"Unknown dataset: {name}")
    return ds


def apply_split(data, mode: str = "public", labels_per_class: int = 5,
                split_seed: int = 0, val_size: int = 500, test_size: int = 1000,
                **kwargs):
    if mode == "public":
        if not hasattr(data, "train_mask") or data.train_mask is None:
            raise ValueError("Dataset has no public split; use mode='low_label'.")
        # WebKB ships multiple splits as [N, 10] masks — take column split_seed % 10
        if data.train_mask.dim() == 2:
            col = split_seed % data.train_mask.size(1)
            data.train_mask = data.train_mask[:, col]
            data.val_mask = data.val_mask[:, col]
            data.test_mask = data.test_mask[:, col]
        return data

    if mode == "ratio":
        # HGCN protocol for datasets without a public split (Disease):
        # seeded stratified 30/10/60 by default.
        g = torch.Generator().manual_seed(split_seed)
        n, y = data.num_nodes, data.y
        train_ratio = kwargs.get("train_ratio", 0.30)
        val_ratio = kwargs.get("val_ratio", 0.10)
        train_mask = torch.zeros(n, dtype=torch.bool)
        val_mask = torch.zeros(n, dtype=torch.bool)
        test_mask = torch.zeros(n, dtype=torch.bool)
        for c in y.unique():  # stratified: preserve class balance per split
            idx = (y == c).nonzero(as_tuple=True)[0]
            perm = idx[torch.randperm(len(idx), generator=g)]
            n_tr = max(1, int(round(train_ratio * len(perm))))
            n_va = max(1, int(round(val_ratio * len(perm))))
            train_mask[perm[:n_tr]] = True
            val_mask[perm[n_tr:n_tr + n_va]] = True
            test_mask[perm[n_tr + n_va:]] = True
        data.train_mask, data.val_mask, data.test_mask = train_mask, val_mask, test_mask
        return data

    if mode == "low_label":
        g = torch.Generator().manual_seed(split_seed)
        n, y = data.num_nodes, data.y
        train_mask = torch.zeros(n, dtype=torch.bool)
        for c in y.unique():
            idx = (y == c).nonzero(as_tuple=True)[0]
            perm = idx[torch.randperm(len(idx), generator=g)]
            train_mask[perm[:labels_per_class]] = True
        rest = (~train_mask).nonzero(as_tuple=True)[0]
        rest = rest[torch.randperm(len(rest), generator=g)]
        val_mask = torch.zeros(n, dtype=torch.bool)
        test_mask = torch.zeros(n, dtype=torch.bool)
        val_mask[rest[:val_size]] = True
        test_mask[rest[val_size:val_size + test_size]] = True
        data.train_mask, data.val_mask, data.test_mask = train_mask, val_mask, test_mask
        return data

    raise ValueError(f"Unknown split mode: {mode}")


Writing /content/src/nemaq/data/loader.py


In [7]:
%%writefile /content/src/nemaq/data/hyperbolicity.py
"""Sampled Gromov delta-hyperbolicity.

delta = 0 for trees; larger delta = less tree-like. Exact computation is
O(n^4), so we use the standard sampled 4-point condition on shortest-path
distances within the largest connected component. This produces the
dataset-stratification table that hypothesis H1 depends on.
"""
import networkx as nx
import numpy as np


def _four_point_delta(d, quad) -> float:
    a, b, c, x = quad
    s1 = d[a][b] + d[c][x]
    s2 = d[a][c] + d[b][x]
    s3 = d[a][x] + d[b][c]
    top2 = sorted((s1, s2, s3))[-2:]
    return (top2[1] - top2[0]) / 2.0


def sampled_gromov_delta(edge_index, num_nodes: int, n_samples: int = 5000,
                         max_component: int = 3000, seed: int = 0) -> dict:
    """Return {'delta_mean', 'delta_max', 'n_samples'} on sampled 4-tuples."""
    rng = np.random.default_rng(seed)
    g = nx.Graph()
    g.add_nodes_from(range(num_nodes))
    g.add_edges_from(edge_index.t().tolist())
    comp = max(nx.connected_components(g), key=len)
    nodes = list(comp)
    if len(nodes) > max_component:
        nodes = list(rng.choice(nodes, size=max_component, replace=False))
    sub = g.subgraph(nodes)
    d = dict(nx.all_pairs_shortest_path_length(sub))

    deltas = []
    node_arr = np.array(list(sub.nodes))
    while len(deltas) < n_samples:
        quad = rng.choice(node_arr, size=4, replace=False)
        try:
            deltas.append(_four_point_delta(d, quad))
        except KeyError:
            continue  # disconnected pair within sampled subgraph
    deltas = np.array(deltas)
    return {
        "delta_mean": float(deltas.mean()),
        "delta_max": float(deltas.max()),
        "n_samples": len(deltas),
    }


Writing /content/src/nemaq/data/hyperbolicity.py


In [8]:
%%writefile /content/src/nemaq/models/classical.py
"""Classical baselines and the classical bypass branch.

These must reproduce published Cora/Citeseer/Pubmed numbers (Gate 2)
before any hybrid experiment counts.
"""
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GATConv, GCNConv


class GCN(nn.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


class GAT(nn.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int,
                 heads: int = 8, dropout: float = 0.6):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, out_dim, heads=1,
                             concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


class MLP(nn.Module):
    """Graph-blind control: any model beating this proves structure matters."""

    def __init__(self, in_dim: int, hidden: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x, edge_index=None):
        return self.net(x)


class GCNEncoder(nn.Module):
    """Bypass branch: GCN body without classifier head, used inside NEMA-Q."""

    def __init__(self, in_dim: int, hidden: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


Writing /content/src/nemaq/models/classical.py


In [9]:
%%writefile /content/src/nemaq/models/hyperbolic.py
"""Poincare-ball hyperbolic graph encoder (geoopt).

HGCN-style tangent-space message passing:
  1. exp-map features to the ball at the origin,
  2. per-layer: Mobius linear transform, log-map to tangent space,
     neighborhood mean-aggregate, exp-map back, pointwise nonlinearity
     applied in tangent space,
  3. final log-map so downstream fusion operates in a shared tangent space.

Curvature c is learnable (softplus-positive) unless frozen by config —
learned curvature per dataset is itself reportable evidence for H1.
"""
import geoopt
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.utils import add_self_loops, degree


class HypLinear(nn.Module):
    def __init__(self, manifold: geoopt.PoincareBall, in_dim: int, out_dim: int):
        super().__init__()
        self.manifold = manifold
        self.weight = nn.Parameter(torch.empty(out_dim, in_dim))
        self.bias = nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        mv = self.manifold.mobius_matvec(self.weight, x)
        b = self.manifold.expmap0(self.bias)
        return self.manifold.mobius_add(mv, b)


class HypGraphConv(nn.Module):
    """Aggregate in tangent space at the origin (mean over neighbors, incl. self)."""

    def __init__(self, manifold: geoopt.PoincareBall, in_dim: int, out_dim: int,
                 dropout: float = 0.5, act: bool = True):
        super().__init__()
        self.manifold = manifold
        self.lin = HypLinear(manifold, in_dim, out_dim)
        self.dropout = dropout
        self.act = act

    def forward(self, x, edge_index):
        x = self.lin(x)
        t = self.manifold.logmap0(x)
        ei, _ = add_self_loops(edge_index, num_nodes=t.size(0))
        row, col = ei
        deg = degree(row, t.size(0), dtype=t.dtype).clamp(min=1)
        agg = torch.zeros_like(t).index_add_(0, row, t[col]) / deg.unsqueeze(-1)
        if self.act:
            agg = F.relu(agg)
        agg = F.dropout(agg, p=self.dropout, training=self.training)
        return self.manifold.projx(self.manifold.expmap0(agg))


MAX_TANGENT_NORM = 2.5  # keep exp-mapped points away from the ball boundary


def clip_tangent(v: torch.Tensor, max_norm: float = MAX_TANGENT_NORM) -> torch.Tensor:
    norm = v.norm(dim=-1, keepdim=True).clamp(min=1e-12)
    return v * (max_norm / norm).clamp(max=1.0)


class HyperbolicEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int,
                 dropout: float = 0.5, c: float = 1.0, learnable_c: bool = True):
        super().__init__()
        self.manifold = geoopt.PoincareBall(c=c, learnable=learnable_c)
        # Euclidean compression before the exp-map: mapping raw high-dim
        # features (e.g. 1433-dim Cora bags-of-words) straight onto the ball
        # saturates points at the boundary where gradients vanish.
        self.feat = nn.Linear(in_dim, hidden)
        self.conv1 = HypGraphConv(self.manifold, hidden, hidden, dropout)
        self.conv2 = HypGraphConv(self.manifold, hidden, out_dim, dropout, act=False)

    def forward(self, x, edge_index):
        t = clip_tangent(self.feat(x))
        h = self.manifold.projx(self.manifold.expmap0(t))
        h = self.conv1(h, edge_index)
        h = self.conv2(h, edge_index)
        return self.manifold.logmap0(h)  # tangent-space output for fusion

    @property
    def curvature(self) -> float:
        return float(self.manifold.c.detach())


class HGCN(nn.Module):
    """Standalone hyperbolic GNN baseline (ICEQT'26 referee B: 'add ...
    hyperbolic GNN baselines under the same split')."""

    def __init__(self, in_dim: int, hidden: int, out_dim: int,
                 dropout: float = 0.5, c: float = 1.0, learnable_c: bool = True):
        super().__init__()
        self.encoder = HyperbolicEncoder(in_dim, hidden, hidden, dropout,
                                         c=c, learnable_c=learnable_c)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, x, edge_index):
        return self.head(self.encoder(x, edge_index))

    @property
    def curvature(self) -> float:
        return self.encoder.curvature


Writing /content/src/nemaq/models/hyperbolic.py


In [10]:
%%writefile /content/src/nemaq/models/quantum.py
"""Variational quantum branch (PennyLane TorchLayer).

Design constraints from the proposal (barren-plateau mitigation):
  - small and shallow: n_qubits <= 8, depth <= 4 (enforced),
  - angle embedding of a compressed feature vector,
  - StronglyEntanglingLayers ansatz,
  - per-qubit Pauli-Z expectations as output (local observables).

`frozen_random=True` gives the NEMA-R recipe control: same circuit, random
fixed parameters — separates "trainable quantum features" from "any fixed
nonlinear random projection".
"""
import pennylane as qml
import torch
from torch import nn

MAX_QUBITS = 8
MAX_DEPTH = 4


class QuantumBranch(nn.Module):
    def __init__(self, in_dim: int, n_qubits: int = 4, depth: int = 2,
                 out_dim: int = 16, frozen_random: bool = False,
                 shots: int | None = None, seed: int = 0):
        super().__init__()
        assert n_qubits <= MAX_QUBITS, f"n_qubits > {MAX_QUBITS} violates BP budget"
        assert depth <= MAX_DEPTH, f"depth > {MAX_DEPTH} violates BP budget"
        self.n_qubits = n_qubits
        self.depth = depth

        # classical compressor: features -> rotation angles
        self.compress = nn.Linear(in_dim, n_qubits)

        dev = qml.device("default.qubit", wires=n_qubits, shots=shots)

        @qml.qnode(dev, interface="torch", diff_method="backprop" if shots is None else "parameter-shift")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
            qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]

        shape = qml.StronglyEntanglingLayers.shape(n_layers=depth, n_wires=n_qubits)
        self.qlayer = qml.qnn.TorchLayer(circuit, {"weights": shape})

        if frozen_random:
            g = torch.Generator().manual_seed(seed)
            with torch.no_grad():
                self.qlayer.weights.copy_(
                    torch.rand(self.qlayer.weights.shape, generator=g) * 2 * torch.pi
                )
            self.qlayer.weights.requires_grad_(False)

        self.project = nn.Linear(n_qubits, out_dim)

    def forward(self, x, edge_index=None):
        angles = torch.tanh(self.compress(x)) * torch.pi  # bounded angles
        # default.qubit simulates the statevector on CPU; keep the circuit
        # weights and inputs there even when the rest of the model is on GPU,
        # then return to the model device for fusion.
        out_device = angles.device
        if self.qlayer.weights.device.type != "cpu":
            self.qlayer.cpu()
        q = self.qlayer(angles.cpu())
        return self.project(q.to(out_device))

    @property
    def circuit_weights(self) -> torch.Tensor:
        return self.qlayer.weights


Writing /content/src/nemaq/models/quantum.py


In [11]:
%%writefile /content/src/nemaq/models/surrogate.py
"""Dequantization control (NEMA-C): classical branch parameter-matched to the PQC.

The headline quantum claim is NEMA-Q vs NEMA-C — same fusion, same head,
same training recipe; only this branch differs. Parameter matching is
asserted in tests/test_param_match.py, not assumed.
"""
import torch
from torch import nn


def pqc_param_count(n_qubits: int, depth: int) -> int:
    """StronglyEntanglingLayers: depth * n_qubits * 3 rotation angles."""
    return depth * n_qubits * 3


class SurrogateBranch(nn.Module):
    """Mirrors QuantumBranch interface: compress -> bounded nonlinear core -> project.

    The core is a small MLP whose hidden width is solved so its parameter
    count matches the PQC's trainable circuit weights as closely as possible
    (always within one hidden unit's worth; exact count reported).
    """

    def __init__(self, in_dim: int, n_qubits: int = 4, depth: int = 2, out_dim: int = 16):
        super().__init__()
        self.compress = nn.Linear(in_dim, n_qubits)  # identical to quantum branch
        target = pqc_param_count(n_qubits, depth)
        # core: Linear(n_qubits->h) + Linear(h->n_qubits), params = h*(2*n_qubits+1)+n_qubits
        h = max(1, round((target - n_qubits) / (2 * n_qubits + 1)))
        self.core = nn.Sequential(
            nn.Linear(n_qubits, h), nn.Tanh(), nn.Linear(h, n_qubits), nn.Tanh(),
        )
        self.project = nn.Linear(n_qubits, out_dim)  # identical to quantum branch

    def forward(self, x, edge_index=None):
        angles = torch.tanh(self.compress(x)) * torch.pi
        return self.project(self.core(angles))

    def core_param_count(self) -> int:
        return sum(p.numel() for p in self.core.parameters())


Writing /content/src/nemaq/models/surrogate.py


In [12]:
%%writefile /content/src/nemaq/models/fusion.py
"""Fusion of branch outputs. Two modes (config: model.fusion_mode):

- "residual" (default): the classical bypass is the trunk; every other branch
  enters as a sigmoid-gated residual whose gate bias starts at -2
  (sigma(-2) ~ 0.12), so training begins from "approximately the bypass GCN"
  and exotic branches must earn their gate mass. This enforces the proposal's
  graceful-degradation claim architecturally: worst case ~= plain GCN.

- "softmax": all branches compete through a softmax gate (zero-init = uniform
  start). Kept as an ablation — the Phase-4 matrix showed naive softmax
  competition is unstable (test acc 0.32–0.78 across seeds) regardless of
  branch type, classical surrogate included.

Gate values are exposed (`last_gates`) for telemetry and H4.
"""
import torch
from torch import nn

RESIDUAL_GATE_BIAS_INIT = -2.0


class GatedFusion(nn.Module):
    def __init__(self, branch_dims: dict[str, int], fused_dim: int,
                 mode: str = "residual", trunk: str = "bypass",
                 gate_bias_init: float = RESIDUAL_GATE_BIAS_INIT):
        super().__init__()
        if mode not in ("residual", "softmax"):
            raise ValueError(f"fusion_mode={mode}")
        if mode == "residual" and trunk not in branch_dims:
            raise ValueError(f"residual fusion needs trunk branch '{trunk}'")
        self.mode = mode
        self.trunk = trunk
        self.branch_names = list(branch_dims)
        self.residual_names = [n for n in self.branch_names if n != trunk]
        self.proj = nn.ModuleDict(
            {name: nn.Linear(d, fused_dim) for name, d in branch_dims.items()}
        )
        n_gated = len(self.residual_names) if mode == "residual" else len(self.branch_names)
        if n_gated > 0:
            self.gate = nn.Linear(fused_dim * len(branch_dims), n_gated)
            nn.init.zeros_(self.gate.weight)
            nn.init.constant_(self.gate.bias,
                              gate_bias_init if mode == "residual" else 0.0)
        else:  # trunk-only ablation: nothing to gate
            self.gate = None
        self.last_gates: torch.Tensor | None = None  # [N, n_gated], telemetry

    def forward(self, branch_outputs: dict[str, torch.Tensor]) -> torch.Tensor:
        projected = {n: self.proj[n](branch_outputs[n]) for n in self.branch_names}
        if self.gate is None:
            return projected[self.trunk]
        gate_in = torch.cat([projected[n] for n in self.branch_names], dim=-1)

        if self.mode == "softmax":
            gates = torch.softmax(self.gate(gate_in), dim=-1)      # [N, B]
            self.last_gates = gates.detach()
            stacked = torch.stack([projected[n] for n in self.branch_names], dim=1)
            return (gates.unsqueeze(-1) * stacked).sum(dim=1)

        gates = torch.sigmoid(self.gate(gate_in))                  # [N, B-1]
        self.last_gates = gates.detach()
        fused = projected[self.trunk]
        for i, name in enumerate(self.residual_names):
            fused = fused + gates[:, i:i + 1] * projected[name]
        return fused

    def gate_dict(self) -> dict[str, torch.Tensor]:
        """Map telemetry gate columns to branch names."""
        if self.gate is None or self.last_gates is None:
            return {}
        names = self.residual_names if self.mode == "residual" else self.branch_names
        return {n: self.last_gates[:, i] for i, n in enumerate(names)}


Writing /content/src/nemaq/models/fusion.py


In [13]:
%%writefile /content/src/nemaq/models/nemaq.py
"""NEMA-Q assembly + model factory.

All ablations are config swaps (design rule #2):
  geometry:     "hyperbolic" | "euclidean"          (H1 control)
  quantum_mode: "pqc" | "surrogate" | "frozen_random" | "off"  (H2 controls)
"""
import torch
from torch import nn

from .classical import GAT, GCN, MLP, GCNEncoder
from .fusion import GatedFusion
from .hyperbolic import HGCN, HyperbolicEncoder
from .quantum import QuantumBranch
from .surrogate import SurrogateBranch


class NemaQ(nn.Module):
    def __init__(self, in_dim: int, num_classes: int, cfg: dict):
        super().__init__()
        hid = cfg.get("hidden", 64)
        emb = cfg.get("branch_dim", 16)
        fused = cfg.get("fused_dim", 32)
        dropout = cfg.get("dropout", 0.5)
        geometry = cfg.get("geometry", "hyperbolic")
        qmode = cfg.get("quantum_mode", "pqc")

        self.branches = nn.ModuleDict()
        if geometry == "hyperbolic":
            self.branches["geo"] = HyperbolicEncoder(
                in_dim, hid, emb, dropout,
                c=cfg.get("curvature", 1.0),
                learnable_c=cfg.get("learnable_curvature", True),
            )
        elif geometry == "euclidean":
            self.branches["geo"] = GCNEncoder(in_dim, hid, emb, dropout)
        elif geometry != "off":  # "off": trunk-only diagnostic ablation
            raise ValueError(f"geometry={geometry}")

        nq, depth = cfg.get("n_qubits", 4), cfg.get("q_depth", 2)
        if qmode == "pqc":
            self.branches["q"] = QuantumBranch(in_dim, nq, depth, emb,
                                               shots=cfg.get("shots"))
        elif qmode == "frozen_random":
            self.branches["q"] = QuantumBranch(in_dim, nq, depth, emb,
                                               frozen_random=True,
                                               seed=cfg.get("q_seed", 0))
        elif qmode == "surrogate":
            self.branches["q"] = SurrogateBranch(in_dim, nq, depth, emb)
        elif qmode != "off":
            raise ValueError(f"quantum_mode={qmode}")

        self.branches["bypass"] = GCNEncoder(in_dim, hid, emb, dropout)

        self.fusion = GatedFusion({n: emb for n in self.branches}, fused,
                                  mode=cfg.get("fusion_mode", "residual"),
                                  gate_bias_init=cfg.get("gate_bias_init", -2.0))
        self.head = nn.Linear(fused, num_classes)
        # Per-branch auxiliary classifiers (deep supervision): each branch
        # learns discriminative features regardless of gate dynamics —
        # prevents the gate-collapse branch starvation seen with pure
        # softmax fusion.
        self.aux_heads = nn.ModuleDict(
            {n: nn.Linear(emb, num_classes) for n in self.branches}
        )
        self.last_aux_logits: dict[str, torch.Tensor] = {}
        self.last_branch_outputs: dict[str, torch.Tensor] = {}  # telemetry

    def forward(self, x, edge_index, disable_branch: str | None = None,
                branch_override: dict[str, torch.Tensor] | None = None):
        """`disable_branch` zeroes one branch output (leave-branch-out);
        `branch_override` substitutes a branch's output tensor (used by QOA
        to differentiate through the observable vector)."""
        outs = {}
        for name, branch in self.branches.items():
            if branch_override and name in branch_override:
                o = branch_override[name]
            else:
                o = branch(x, edge_index)
            if name == disable_branch:
                o = torch.zeros_like(o)
            outs[name] = o
        self.last_branch_outputs = {k: v.detach() for k, v in outs.items()}
        self.last_aux_logits = {n: self.aux_heads[n](outs[n]) for n in self.branches}
        return self.head(self.fusion(outs))


def build_model(name: str, in_dim: int, num_classes: int, cfg: dict) -> nn.Module:
    name = name.lower()
    hid = cfg.get("hidden", 64)
    dropout = cfg.get("dropout", 0.5)
    if name == "gcn":
        return GCN(in_dim, hid, num_classes, dropout)
    if name == "gat":
        return GAT(in_dim, cfg.get("gat_hidden", 8), num_classes,
                   cfg.get("heads", 8), cfg.get("gat_dropout", 0.6))
    if name == "mlp":
        return MLP(in_dim, hid, num_classes, dropout)
    if name == "hgcn":
        return HGCN(in_dim, hid, num_classes, dropout,
                    c=cfg.get("curvature", 1.0),
                    learnable_c=cfg.get("learnable_curvature", True))
    if name == "nemaq":
        return NemaQ(in_dim, num_classes, cfg)
    raise ValueError(f"Unknown model: {name}")


Writing /content/src/nemaq/models/nemaq.py


In [14]:
%%writefile /content/src/nemaq/telemetry/gradients.py
"""Per-branch gradient telemetry — the barren-plateau instrument (H3).

After each backward pass, records per-branch gradient L2 norm and
per-parameter gradient variance. For the quantum branch, circuit weights are
additionally tracked in isolation (``circuit_grad_var``): pooling them with
the large classical compress/project layers drowns the barren-plateau signal
in near-zero classical gradients.

H3 criterion (relative, not absolute): the circuit-weight gradient variance
must stay within ``H3_RELATIVE_ORDERS`` orders of magnitude of the best
classical branch's variance. Absolute floors from the BP literature assume
variance over random inits of circuit params only and do not transfer to a
trained mixed-device hybrid; the classical branches provide the matched
"healthy gradient" reference at the same loss scale.
"""
from collections import defaultdict

import torch
from torch import nn

H3_RELATIVE_ORDERS = 2.0  # circuit var may lag classical var by <= 2 orders of magnitude


class GradientTelemetry:
    def __init__(self, model: nn.Module, branch_attr: str = "branches"):
        self.model = model
        self.branch_attr = branch_attr
        self.history: dict[str, list[dict]] = defaultdict(list)

    def _branch_modules(self) -> dict[str, nn.Module]:
        branches = getattr(self.model, self.branch_attr, None)
        if branches is not None:
            return dict(branches.items())
        return {"model": self.model}

    @staticmethod
    def _grad_stats(params) -> dict | None:
        # .cpu(): branches may live on different devices (PQC is CPU-pinned
        # while classical branches sit on GPU); stats are scalars anyway.
        grads = [p.grad.flatten().cpu() for p in params
                 if p.grad is not None and p.requires_grad]
        if not grads:
            return None
        g = torch.cat(grads)
        return {
            "grad_norm": float(g.norm()),
            "grad_var": float(g.var(unbiased=False)),
            "grad_absmean": float(g.abs().mean()),
        }

    @torch.no_grad()
    def record(self, step: int) -> dict[str, dict]:
        """Call after loss.backward(), before optimizer.step()."""
        snapshot = {}
        for name, module in self._branch_modules().items():
            entry = self._grad_stats(module.parameters())
            if entry is None:
                continue
            entry["step"] = step
            # isolate PQC circuit weights (QuantumBranch exposes them)
            cw = getattr(module, "circuit_weights", None)
            if cw is not None and cw.grad is not None:
                entry["circuit_grad_var"] = float(
                    cw.grad.flatten().cpu().var(unbiased=False))
            self.history[name].append(entry)
            snapshot[name] = entry
        return snapshot

    def summary(self) -> dict:
        out = {}
        for name, entries in self.history.items():
            variances = [e["grad_var"] for e in entries]
            s = {
                "final_grad_var": variances[-1],
                "min_grad_var": min(variances),
                "n_steps": len(variances),
            }
            circuit = [e["circuit_grad_var"] for e in entries
                       if "circuit_grad_var" in e]
            if circuit:
                s["min_circuit_grad_var"] = min(circuit)
                s["final_circuit_grad_var"] = circuit[-1]
            out[name] = s

        # H3 verdict: circuit variance vs best classical-branch variance
        circuit_branches = {n: s for n, s in out.items()
                            if "min_circuit_grad_var" in s}
        classical = [s["min_grad_var"] for n, s in out.items()
                     if n not in circuit_branches]
        if circuit_branches and classical:
            ref = max(classical)
            for n, s in circuit_branches.items():
                ratio_ok = s["min_circuit_grad_var"] >= ref * 10 ** (-H3_RELATIVE_ORDERS)
                s["h3_pass_relative"] = bool(ratio_ok)
                s["h3_classical_reference_var"] = ref
        return out


Writing /content/src/nemaq/telemetry/gradients.py


In [15]:
%%writefile /content/src/nemaq/telemetry/expressibility.py
"""Circuit diagnostics: expressibility (Sim et al. 2019) and
Meyer-Wallach entanglement capability.

Expressibility: KL divergence between the fidelity distribution of states
prepared with random parameter pairs and the Haar-random fidelity
distribution P_Haar(F) = (2^n - 1)(1 - F)^(2^n - 2). Lower KL = more
expressive. Reported per (n_qubits, depth) cell of the sweep.
"""
import numpy as np
import pennylane as qml


def _random_state(n_qubits: int, depth: int, rng: np.random.Generator) -> np.ndarray:
    dev = qml.device("default.qubit", wires=n_qubits)
    shape = qml.StronglyEntanglingLayers.shape(n_layers=depth, n_wires=n_qubits)
    weights = rng.uniform(0, 2 * np.pi, size=shape)
    inputs = rng.uniform(0, 2 * np.pi, size=n_qubits)

    @qml.qnode(dev)
    def circuit():
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return qml.state()

    return circuit()


def expressibility_kl(n_qubits: int, depth: int, n_pairs: int = 2000,
                      n_bins: int = 75, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    fids = np.empty(n_pairs)
    for i in range(n_pairs):
        s1 = _random_state(n_qubits, depth, rng)
        s2 = _random_state(n_qubits, depth, rng)
        fids[i] = np.abs(np.vdot(s1, s2)) ** 2

    edges = np.linspace(0, 1, n_bins + 1)
    hist, _ = np.histogram(fids, bins=edges)
    p = hist / hist.sum()

    d = 2 ** n_qubits
    centers = (edges[:-1] + edges[1:]) / 2
    haar = (d - 1) * (1 - centers) ** (d - 2)
    q = haar / haar.sum()

    mask = p > 0
    return float(np.sum(p[mask] * np.log(p[mask] / np.maximum(q[mask], 1e-12))))


def meyer_wallach(n_qubits: int, depth: int, n_samples: int = 200,
                  seed: int = 0) -> float:
    """Q in [0, 1]; 0 = product states, 1 = maximal average entanglement."""
    rng = np.random.default_rng(seed)
    qs = []
    for _ in range(n_samples):
        state = _random_state(n_qubits, depth, rng)
        psi = state.reshape([2] * n_qubits)
        purity_sum = 0.0
        for k in range(n_qubits):
            m = np.moveaxis(psi, k, 0).reshape(2, -1)
            rho = m @ m.conj().T
            purity_sum += np.real(np.trace(rho @ rho))
        qs.append(2 * (1 - purity_sum / n_qubits))
    return float(np.mean(qs))


Writing /content/src/nemaq/telemetry/expressibility.py


In [16]:
%%writefile /content/src/nemaq/telemetry/attribution.py
"""Per-node, per-branch attribution.

Two instruments:
  1. leave_branch_out — per-node change in predicted-class logit when one
     branch is zeroed. Ground truth for branch usefulness.
  2. gate_attribution — fusion gate weight per node per branch. Cheap proxy.

H4 = Spearman correlation between (1) and (2); computed in analysis.
"""
import torch

from scipy.stats import spearmanr


@torch.no_grad()
def leave_branch_out(model, x, edge_index, mask=None) -> dict[str, torch.Tensor]:
    """Return per-branch tensor of logit drops (positive = branch helped)."""
    model.eval()
    full = model(x, edge_index)
    pred = full.argmax(dim=-1)
    idx = torch.arange(full.size(0), device=full.device)
    full_logit = full[idx, pred]

    deltas = {}
    for name in model.branches:
        ablated = model(x, edge_index, disable_branch=name)
        drop = full_logit - ablated[idx, pred]
        deltas[name] = drop[mask] if mask is not None else drop
    return deltas


@torch.no_grad()
def gate_attribution(model, x, edge_index, mask=None) -> dict[str, torch.Tensor]:
    """Note: in residual fusion mode the trunk branch has no gate and is
    absent from the result (it is always fully present by construction)."""
    model.eval()
    model(x, edge_index)
    out = {}
    for name, g in model.fusion.gate_dict().items():
        out[name] = g[mask] if mask is not None else g
    return out


def h4_correlation(model, x, edge_index, mask=None) -> dict[str, float]:
    """Spearman rho between gate weight and leave-branch-out delta, per
    gated branch (trunk excluded in residual mode)."""
    lbo = leave_branch_out(model, x, edge_index, mask)
    gates = gate_attribution(model, x, edge_index, mask)
    return {
        name: float(spearmanr(gates[name].cpu().numpy(),
                              lbo[name].cpu().numpy()).statistic)
        for name in gates
    }


Writing /content/src/nemaq/telemetry/attribution.py


In [17]:
%%writefile /content/src/nemaq/telemetry/qoa.py
"""Quantum Observable Attribution (QOA) + faithfulness validation.

QOA (the ICEQT'26 paper's core method): per-node grad x input of the
predicted-class logit with respect to the measured quantum observables
(per-qubit Pauli-Z expectations), i.e. which observables drive the decision.

Referee B (ICEQT'26): "QOA is gradient saliency over intermediate
observables, but the paper does not provide faithfulness tests,
randomization checks, perturbation tests." This module supplies exactly
those three checks:

  1. faithfulness_masking  — zeroing the top-attributed observable must hurt
     the predicted logit more than zeroing the bottom-attributed one.
  2. perturbation_test     — noise injected on high-attribution observables
     must change predictions more than on low-attribution ones.
  3. randomization_check   — attributions must decay when the downstream
     model (fusion + head) is re-randomized (Adebayo et al. 2018 sanity check).
"""
import copy

import torch

from scipy.stats import spearmanr


def _observables(model, x):
    """Recompute the PQC observable vector [N, n_qubits] outside the graph."""
    qb = model.branches["q"]
    with torch.no_grad():
        angles = torch.tanh(qb.compress(x)) * torch.pi
        obs = qb.qlayer(angles.cpu()).to(x.device)
    return obs


def _logits_from_obs(model, x, edge_index, obs):
    out_q = model.branches["q"].project(obs)
    return model(x, edge_index, branch_override={"q": out_q})


def observable_attribution(model, x, edge_index):
    """Return (attr [N, n_qubits], obs [N, n_qubits]): grad x input of the
    predicted-class logit w.r.t. each measured observable."""
    model.eval()
    obs = _observables(model, x).requires_grad_(True)
    # enable_grad: callers (masking / perturbation checks) run under
    # @torch.no_grad(), which would otherwise leave sel without a graph.
    with torch.enable_grad():
        logits = _logits_from_obs(model, x, edge_index, obs)
        pred = logits.argmax(dim=-1)
        sel = logits[torch.arange(len(pred), device=logits.device), pred].sum()
        (grad,) = torch.autograd.grad(sel, obs)
    return (grad * obs).detach(), obs.detach()


@torch.no_grad()
def faithfulness_masking(model, x, edge_index, k: int = 1, mask=None) -> dict:
    """Zero top-k vs bottom-k attributed observables per node; report mean
    predicted-logit drop. Faithful attribution => drop_top >> drop_bottom."""
    attr, obs = observable_attribution(model, x, edge_index)
    base = _logits_from_obs(model, x, edge_index, obs)
    pred = base.argmax(dim=-1)
    idx = torch.arange(len(pred), device=base.device)
    order = attr.abs().argsort(dim=1)

    drops = {}
    for name, cols in {"top": order[:, -k:], "bottom": order[:, :k]}.items():
        obs_m = obs.clone().scatter_(1, cols, 0.0)
        masked = _logits_from_obs(model, x, edge_index, obs_m)
        d = base[idx, pred] - masked[idx, pred]
        drops[name] = float(d[mask].mean() if mask is not None else d.mean())
    drops["faithful"] = drops["top"] > drops["bottom"]
    return drops


@torch.no_grad()
def perturbation_test(model, x, edge_index, sigma: float = 0.25,
                      k: int = 1, mask=None, seed: int = 0) -> dict:
    """Gaussian noise on top-k vs bottom-k attributed observables; report
    prediction flip rates. Faithful => flip_top > flip_bottom."""
    attr, obs = observable_attribution(model, x, edge_index)
    base_pred = _logits_from_obs(model, x, edge_index, obs).argmax(dim=-1)
    order = attr.abs().argsort(dim=1)
    g = torch.Generator(device="cpu").manual_seed(seed)

    flips = {}
    for name, cols in {"top": order[:, -k:], "bottom": order[:, :k]}.items():
        noise = torch.randn(cols.shape, generator=g).to(obs.device) * sigma
        obs_p = obs.clone().scatter_add_(1, cols, noise)
        pred = _logits_from_obs(model, x, edge_index, obs_p).argmax(dim=-1)
        f = (pred != base_pred).float()
        flips[name] = float(f[mask].mean() if mask is not None else f.mean())
    flips["faithful"] = flips["top"] > flips["bottom"]
    return flips


def randomization_check(model, x, edge_index, seed: int = 0) -> dict:
    """Re-randomize fusion + head; Spearman correlation between original and
    randomized attributions should collapse toward 0 (attribution depends on
    the learned model, not on architecture artifacts)."""
    attr_orig, _ = observable_attribution(model, x, edge_index)

    # the last forward leaves non-leaf tensors (aux logits / branch outputs)
    # cached on the module; deepcopy chokes on them — clear first.
    if hasattr(model, "last_aux_logits"):
        model.last_aux_logits = {}
    if hasattr(model, "last_branch_outputs"):
        model.last_branch_outputs = {}
    rand = copy.deepcopy(model)
    torch.manual_seed(seed)
    for module in (rand.fusion, rand.head):
        for p in module.parameters():
            if p.dim() > 1:
                torch.nn.init.xavier_uniform_(p)
            else:
                torch.nn.init.normal_(p, std=0.1)
    attr_rand, _ = observable_attribution(rand, x, edge_index)

    rho = spearmanr(attr_orig.flatten().cpu().numpy(),
                    attr_rand.flatten().cpu().numpy()).statistic
    return {"spearman_orig_vs_randomized": float(rho),
            "passes": abs(float(rho)) < 0.5}


Writing /content/src/nemaq/telemetry/qoa.py


In [18]:
%%writefile /content/src/nemaq/training/trainer.py
"""Single-run trainer: full-batch node classification with early stopping,
gradient telemetry always on (instrumentation-first), manifest on exit.

Note: geoopt manifold parameters (learnable curvature) train correctly with
a Riemannian optimizer; we use geoopt.optim.RiemannianAdam, which reduces to
Adam for Euclidean parameters — one optimizer for every model variant.
"""
import copy
from pathlib import Path

import geoopt
import torch
import torch.nn.functional as F

from ..models import build_model
from ..telemetry.gradients import GradientTelemetry
from ..utils.manifest import write_manifest
from ..utils.seed import set_seed


def accuracy(logits, y, mask) -> float:
    return float((logits[mask].argmax(dim=-1) == y[mask]).float().mean())


def f1_per_class(logits, y, mask) -> list[float]:
    """Per-class F1 — min over classes is the minority-class-collapse
    detector (ICEQT'26 referee B flagged an unnoticed per-class F1 collapse)."""
    pred, yt = logits[mask].argmax(dim=-1), y[mask]
    scores = []
    for c in range(int(y.max()) + 1):
        tp = int(((pred == c) & (yt == c)).sum())
        fp = int(((pred == c) & (yt != c)).sum())
        fn = int(((pred != c) & (yt == c)).sum())
        denom = 2 * tp + fp + fn
        scores.append(2 * tp / denom if denom > 0 else 0.0)
    return scores


def param_counts(model) -> dict[str, int]:
    counts = {"total": sum(p.numel() for p in model.parameters() if p.requires_grad)}
    if hasattr(model, "branches"):
        for name, b in model.branches.items():
            counts[f"branch_{name}"] = sum(p.numel() for p in b.parameters()
                                           if p.requires_grad)
    return counts


def train_run(cfg: dict, data, seed: int, run_dir: str | Path,
              return_model: bool = False):
    set_seed(seed, deterministic=cfg.get("deterministic", True))
    device = torch.device(cfg.get("device", "cuda" if torch.cuda.is_available() else "cpu"))
    data = data.to(device)

    model = build_model(
        cfg["model"]["name"], data.num_features,
        int(data.y.max()) + 1, cfg["model"],
    ).to(device)

    # No weight decay on fusion gates (decay drags them back toward their
    # init and fights gate learning) or manifold curvature.
    decay, no_decay = [], []
    for pname, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if ("fusion.gate" in pname or "manifold" in pname)
         else decay).append(p)
    opt = geoopt.optim.RiemannianAdam(
        [{"params": decay, "weight_decay": cfg["train"].get("weight_decay", 5e-4)},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=cfg["train"].get("lr", 0.01),
    )
    telemetry = GradientTelemetry(model)
    aux_weight = cfg["train"].get("aux_weight", 0.3)
    aux_anneal = cfg["train"].get("aux_anneal", True)

    best_val, best_state, patience_left = -1.0, None, cfg["train"].get("patience", 100)
    max_epochs = cfg["train"].get("epochs", 500)

    for epoch in range(max_epochs):
        model.train()
        opt.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        # deep supervision: every branch trains against the labels directly.
        # Annealed to zero so it prevents early gate-collapse starvation but
        # stops fighting the fused objective at convergence.
        w = aux_weight * (1 - epoch / max_epochs) if aux_anneal else aux_weight
        if w > 0 and getattr(model, "last_aux_logits", None):
            aux = sum(
                F.cross_entropy(a[data.train_mask], data.y[data.train_mask])
                for a in model.last_aux_logits.values()
            ) / len(model.last_aux_logits)
            loss = loss + w * aux
        loss.backward()
        telemetry.record(epoch)
        opt.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            val_acc = accuracy(logits, data.y, data.val_mask)
        if val_acc > best_val:
            best_val = val_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_left = cfg["train"].get("patience", 100)
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
    f1s = f1_per_class(logits, data.y, data.test_mask)
    metrics = {
        "val_acc": best_val,
        "test_acc": accuracy(logits, data.y, data.test_mask),
        "test_macro_f1": sum(f1s) / len(f1s),
        "test_min_class_f1": min(f1s),   # 0.0 here = minority-class collapse
        "epochs_run": epoch + 1,
        "param_counts": param_counts(model),
    }
    if hasattr(model, "branches") and "geo" in model.branches \
            and hasattr(model.branches["geo"], "curvature"):
        metrics["learned_curvature"] = model.branches["geo"].curvature

    write_manifest(run_dir, cfg, seed, metrics, telemetry.summary())
    if return_model:
        return metrics, model, data
    return metrics


Writing /content/src/nemaq/training/trainer.py


In [19]:
%%writefile /content/src/nemaq/analysis/stats.py
"""Pre-registered statistical protocol (addresses gap G2).

- Paired comparisons across seeds (same seeds/splits for both models).
- Wilcoxon signed-rank (non-parametric, n=10 seeds).
- Holm-Bonferroni correction over the hypothesis family.
- Cohen's d on paired differences as effect size.

Analysis consumes run manifests only (design rule #1).
"""
from collections import defaultdict

import numpy as np
from scipy import stats

from ..utils.manifest import read_manifests


def collect_scores(runs_root: str, metric: str = "test_acc") -> dict[tuple, dict[int, float]]:
    """Group manifest metrics by (dataset, model, split_mode), keyed by seed."""
    grouped: dict[tuple, dict[int, float]] = defaultdict(dict)
    for m in read_manifests(runs_root):
        cfg = m["config"]
        model_key = cfg["model"]["name"]
        if model_key == "nemaq":  # variant flags only matter for the hybrid
            model_key += (":" + cfg["model"].get("quantum_mode", "-")
                          + ":" + cfg["model"].get("geometry", "-")
                          + ":" + cfg["model"].get("fusion_mode", "residual"))
        key = (
            cfg["data"]["name"],
            model_key,
            cfg["data"].get("split", "public"),
        )
        grouped[key][m["seed"]] = m["metrics"][metric]
    return grouped


def paired_comparison(scores_a: dict[int, float], scores_b: dict[int, float]) -> dict:
    """Wilcoxon + Cohen's d on seed-paired scores. a = treatment, b = control."""
    seeds = sorted(set(scores_a) & set(scores_b))
    if len(seeds) < 5:
        raise ValueError(f"Only {len(seeds)} paired seeds; protocol requires >= 5 (target 10).")
    a = np.array([scores_a[s] for s in seeds])
    b = np.array([scores_b[s] for s in seeds])
    diff = a - b
    if np.allclose(diff, 0):
        w_p = 1.0
    else:
        w_p = float(stats.wilcoxon(a, b).pvalue)
    sd = diff.std(ddof=1)
    if sd > 0:
        d = float(diff.mean() / sd)
    else:
        # constant nonzero difference = deterministic effect, not zero effect
        d = float(np.sign(diff.mean()) * np.inf) if diff.mean() != 0 else 0.0
    return {
        "n_seeds": len(seeds),
        "mean_a": float(a.mean()), "std_a": float(a.std(ddof=1)),
        "mean_b": float(b.mean()), "std_b": float(b.std(ddof=1)),
        "mean_diff": float(diff.mean()),
        "wilcoxon_p": w_p,
        "cohens_d": d,
    }


def geometry_interaction(runs_root: str, delta_map: dict[str, float],
                         hyp_key: str = "nemaq:pqc:hyperbolic:residual",
                         euc_key: str = "nemaq:pqc:euclidean:residual") -> dict:
    """H1's pooled-power test for the geometry effect.

    The single-dataset hyperbolic-vs-Euclidean comparison is underpowered at
    n=10 seeds (Cora Phase 4: paired d=0.44 -> ~40 seeds needed alone). The
    registered remedy pools evidence instead:
      1. per-dataset paired Wilcoxon (one-sided, H1 direction: hyp > euc),
      2. Stouffer's Z across datasets (pooled directional evidence),
      3. Spearman(delta_mean, gain) across datasets (the trend H1 predicts:
         gain shrinks as delta grows).
    delta_map: {dataset: delta_mean}, from the EDA stats json.
    """
    grouped = collect_scores(runs_root)
    per_dataset, zs = {}, []
    for ds, delta in delta_map.items():
        sh = se = None
        for (d, mk, _sp), scores in grouped.items():
            if d != ds:
                continue
            if mk == hyp_key:
                sh = scores
            elif mk == euc_key:
                se = scores
        if sh is None or se is None:
            continue
        seeds = sorted(set(sh) & set(se))
        if len(seeds) < 5:
            continue
        a = np.array([sh[s] for s in seeds])
        b = np.array([se[s] for s in seeds])
        if np.allclose(a - b, 0):
            p_one = 0.5
        else:
            p_one = float(stats.wilcoxon(a, b, alternative="greater").pvalue)
        cmp = paired_comparison(sh, se)
        cmp.update({"delta_mean": delta, "wilcoxon_p_one_sided": p_one})
        per_dataset[ds] = cmp
        zs.append(stats.norm.isf(p_one))
    if len(per_dataset) < 2:
        raise ValueError("geometry_interaction needs the geometry pair on "
                         ">= 2 datasets")
    z_pooled = float(np.sum(zs) / np.sqrt(len(zs)))
    deltas = [per_dataset[d]["delta_mean"] for d in per_dataset]
    gains = [per_dataset[d]["mean_diff"] for d in per_dataset]
    rho = stats.spearmanr(deltas, gains)
    return {
        "per_dataset": per_dataset,
        "stouffer_z": z_pooled,
        "stouffer_p_one_sided": float(stats.norm.sf(z_pooled)),
        "spearman_delta_gain_rho": float(rho.statistic),
        "spearman_delta_gain_p": float(rho.pvalue),
    }


def holm_bonferroni(pvalues: dict[str, float], alpha: float = 0.05) -> dict[str, dict]:
    """Return per-hypothesis adjusted p and reject decision."""
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    m = len(items)
    out, still_rejecting = {}, True
    running_max = 0.0
    for i, (name, p) in enumerate(items):
        adj = min(1.0, (m - i) * p)
        running_max = max(running_max, adj)  # enforce monotonicity
        reject = still_rejecting and running_max <= alpha
        if not reject:
            still_rejecting = False
        out[name] = {"p": p, "p_adjusted": running_max, "reject_null": reject}
    return out


Writing /content/src/nemaq/analysis/stats.py


In [20]:
%%writefile /content/src/nemaq/models/__init__.py
from .nemaq import build_model  # noqa: F401


Overwriting /content/src/nemaq/models/__init__.py


## 3. Configs

In [21]:
%%writefile /content/configs/base.yaml
# Base config — all experiment configs inherit from this and override.
data:
  name: cora
  root: data
  split: public          # public | low_label
  labels_per_class: 5    # used when split == low_label
  split_seed: 0          # independent of model seed (paired stats)

model:
  name: gcn              # gcn | gat | mlp | nemaq
  hidden: 64
  dropout: 0.5
  # NEMA-Q-specific (ignored by baselines):
  geometry: hyperbolic   # hyperbolic | euclidean
  quantum_mode: pqc      # pqc | surrogate | frozen_random | off
  n_qubits: 4
  q_depth: 2
  branch_dim: 64         # 16 bottlenecked the bypass trunk ~10 pts below plain GCN
  fused_dim: 64
  fusion_mode: residual  # residual (bypass trunk + gated residuals) | softmax (ablation)
  curvature: 1.0
  learnable_curvature: true
  shots: null            # null = exact statevector; int = finite-shot ablation

train:
  lr: 0.01
  weight_decay: 0.0005
  epochs: 500
  patience: 100
  aux_weight: 0.3   # per-branch deep-supervision loss weight (0 disables)
  aux_anneal: true  # linearly decay aux loss to 0 over training: early
                    # anti-starvation without late-training interference

deterministic: true
device: auto


Writing /content/configs/base.yaml


In [22]:
%%writefile /content/configs/cora_gcn.yaml
inherit: base.yaml
model:
  name: gcn


Writing /content/configs/cora_gcn.yaml


In [23]:
%%writefile /content/configs/cora_gat.yaml
inherit: base.yaml
model:
  name: gat


Writing /content/configs/cora_gat.yaml


In [24]:
%%writefile /content/configs/cora_hgcn.yaml
# Standalone hyperbolic GNN baseline (ICEQT'26 review: hyperbolic baselines
# under the same split with reported parameter counts).
inherit: base.yaml
model:
  name: hgcn


Writing /content/configs/cora_hgcn.yaml


In [25]:
%%writefile /content/configs/cora_nemaq.yaml
inherit: base.yaml
model:
  name: nemaq
  geometry: hyperbolic
  quantum_mode: pqc


Writing /content/configs/cora_nemaq.yaml


In [26]:
%%writefile /content/configs/ablations/nemaq_surrogate.yaml
# NEMA-C: dequantization control — the headline comparison partner for NEMA-Q.
inherit: ../base.yaml
model:
  name: nemaq
  geometry: hyperbolic
  quantum_mode: surrogate


Writing /content/configs/ablations/nemaq_surrogate.yaml


In [27]:
%%writefile /content/configs/ablations/nemaq_frozen_random.yaml
# NEMA-R: recipe control — same circuit, frozen random parameters.
inherit: ../base.yaml
model:
  name: nemaq
  geometry: hyperbolic
  quantum_mode: frozen_random


Writing /content/configs/ablations/nemaq_frozen_random.yaml


In [28]:
%%writefile /content/configs/ablations/nemaq_euclidean.yaml
# NEMA-E: geometry control — Euclidean encoder replaces hyperbolic.
inherit: ../base.yaml
model:
  name: nemaq
  geometry: euclidean
  quantum_mode: pqc


Writing /content/configs/ablations/nemaq_euclidean.yaml


In [29]:
%%writefile /content/configs/ablations/nemaq_softmax_fusion.yaml
# Fusion ablation: original softmax-competition gating (known unstable —
# Phase-4 matrix 2026-07-16 showed 0.32-0.78 seed spread). Kept to quantify
# the residual-trunk fix in the paper.
inherit: ../base.yaml
model:
  name: nemaq
  geometry: hyperbolic
  quantum_mode: pqc
  fusion_mode: softmax


Writing /content/configs/ablations/nemaq_softmax_fusion.yaml


In [30]:
%%writefile /content/configs/ablations/nemaq_trunk_only.yaml
# Diagnostic: bypass trunk + fusion + head with NO exotic branches.
# If this fails to reach plain-GCN accuracy (~0.82 Cora), the residual
# scaffold itself (projection/head/recipe) is leaking performance and the
# gap between NEMA-Q and GCN is a scaffold artifact, not a branch effect.
inherit: ../base.yaml
model:
  name: nemaq
  geometry: "off"
  quantum_mode: "off"


Writing /content/configs/ablations/nemaq_trunk_only.yaml


## 4. Tests (Gates 1–3)

In [31]:
%%writefile /content/tests/test_config.py
from pathlib import Path

import pytest

pytest.importorskip("yaml")

from nemaq.utils.config import load_config

CONFIGS = Path(__file__).parent.parent / "configs"


def test_inheritance_merges_overrides():
    cfg = load_config(CONFIGS / "cora_nemaq.yaml")
    assert cfg["model"]["name"] == "nemaq"
    assert cfg["train"]["lr"] == 0.01          # inherited from base
    assert cfg["data"]["name"] == "cora"       # inherited from base


def test_ablation_configs_only_swap_flags():
    base = load_config(CONFIGS / "cora_nemaq.yaml")
    surr = load_config(CONFIGS / "ablations" / "nemaq_surrogate.yaml")
    assert surr["model"]["quantum_mode"] == "surrogate"
    assert surr["train"] == base["train"]      # same recipe — controls differ only in the flag


Writing /content/tests/test_config.py


In [32]:
%%writefile /content/tests/test_stats.py
"""Statistical protocol tests — the analysis code must be trustworthy
before it judges hypotheses."""
import pytest

pytest.importorskip("scipy")

from nemaq.analysis.stats import holm_bonferroni, paired_comparison


def test_paired_comparison_detects_clear_difference():
    a = {s: 0.85 + 0.01 * (s % 3) for s in range(10)}
    b = {s: 0.80 + 0.01 * (s % 3) for s in range(10)}
    r = paired_comparison(a, b)
    assert r["mean_diff"] == pytest.approx(0.05)
    assert r["wilcoxon_p"] < 0.05


def test_paired_comparison_requires_enough_seeds():
    with pytest.raises(ValueError):
        paired_comparison({0: 1.0, 1: 1.0}, {0: 0.9, 1: 0.9})


def test_holm_bonferroni_monotone_and_correct():
    res = holm_bonferroni({"h1": 0.001, "h2": 0.04, "h3": 0.30})
    assert res["h1"]["reject_null"] is True
    assert res["h3"]["reject_null"] is False
    ps = [res[k]["p_adjusted"] for k in ("h1", "h2", "h3")]
    assert ps == sorted(ps)


Writing /content/tests/test_stats.py


In [33]:
%%writefile /content/tests/test_param_match.py
"""Design rule #3: parameter matching between PQC and surrogate is asserted.

If this test fails, the dequantization control is invalid and every
NEMA-Q vs NEMA-C comparison is meaningless.
"""
import pytest

torch = pytest.importorskip("torch")
pytest.importorskip("pennylane")

from nemaq.models.quantum import QuantumBranch
from nemaq.models.surrogate import SurrogateBranch, pqc_param_count

TOLERANCE = 0.15  # surrogate core within 15% of PQC trainable circuit params


@pytest.mark.parametrize("nq,depth", [(4, 1), (4, 2), (6, 2), (8, 4)])
def test_surrogate_matches_pqc(nq, depth):
    q = QuantumBranch(in_dim=32, n_qubits=nq, depth=depth, out_dim=16)
    s = SurrogateBranch(in_dim=32, n_qubits=nq, depth=depth, out_dim=16)

    pqc_params = q.circuit_weights.numel()
    assert pqc_params == pqc_param_count(nq, depth)

    core_params = s.core_param_count()
    rel = abs(core_params - pqc_params) / pqc_params
    assert rel <= TOLERANCE, (
        f"surrogate core {core_params} vs PQC {pqc_params} ({rel:.0%} off)"
    )

    # identical wrapper layers (compress/project) => total diff == core diff
    q_compress = sum(p.numel() for p in q.compress.parameters())
    s_compress = sum(p.numel() for p in s.compress.parameters())
    assert q_compress == s_compress


def test_frozen_random_has_no_trainable_circuit():
    q = QuantumBranch(in_dim=32, n_qubits=4, depth=2, frozen_random=True)
    assert not q.circuit_weights.requires_grad


Writing /content/tests/test_param_match.py


In [34]:
%%writefile /content/tests/test_models.py
"""Component unit tests (Gate 3): shapes, gradient flow, control swaps."""
import pytest

torch = pytest.importorskip("torch")
pytest.importorskip("torch_geometric")

from nemaq.models import build_model

N, F_IN, C = 20, 12, 3


@pytest.fixture
def tiny_graph():
    torch.manual_seed(0)
    x = torch.randn(N, F_IN)
    edge_index = torch.randint(0, N, (2, 60))
    y = torch.randint(0, C, (N,))
    return x, edge_index, y


@pytest.mark.parametrize("name", ["gcn", "gat", "mlp"])
def test_baselines_forward_backward(tiny_graph, name):
    x, ei, y = tiny_graph
    model = build_model(name, F_IN, C, {"hidden": 16})
    out = model(x, ei)
    assert out.shape == (N, C)
    torch.nn.functional.cross_entropy(out, y).backward()
    assert any(p.grad is not None for p in model.parameters())


def test_hyperbolic_encoder_maps_are_inverse():
    geoopt = pytest.importorskip("geoopt")
    from nemaq.models.hyperbolic import HyperbolicEncoder
    enc = HyperbolicEncoder(F_IN, 16, 8)
    v = torch.randn(5, F_IN) * 0.1
    on_ball = enc.manifold.projx(enc.manifold.expmap0(v))
    back = enc.manifold.logmap0(on_ball)
    assert torch.allclose(v, back, atol=1e-4)


@pytest.mark.parametrize("qmode", ["pqc", "surrogate", "frozen_random", "off"])
def test_nemaq_quantum_mode_swaps(tiny_graph, qmode):
    pytest.importorskip("pennylane")
    pytest.importorskip("geoopt")
    x, ei, y = tiny_graph
    cfg = {"hidden": 16, "branch_dim": 8, "fused_dim": 16,
           "n_qubits": 3, "q_depth": 1, "geometry": "euclidean",
           "quantum_mode": qmode}
    model = build_model("nemaq", F_IN, C, cfg)
    out = model(x, ei)
    assert out.shape == (N, C)
    torch.nn.functional.cross_entropy(out, y).backward()


def test_leave_branch_out_changes_output(tiny_graph):
    pytest.importorskip("pennylane")
    x, ei, _ = tiny_graph
    cfg = {"hidden": 16, "branch_dim": 8, "fused_dim": 16,
           "n_qubits": 3, "q_depth": 1, "geometry": "euclidean",
           "quantum_mode": "pqc"}
    model = build_model("nemaq", F_IN, C, cfg)
    model.eval()
    with torch.no_grad():
        full = model(x, ei)
        ablated = model(x, ei, disable_branch="q")
    assert not torch.allclose(full, ablated)


Writing /content/tests/test_models.py


In [35]:
%%writefile /content/tests/test_fusion.py
"""Residual-trunk fusion guarantees: training starts approximately at the
trunk (graceful-degradation floor), softmax ablation starts uniform."""
import pytest

torch = pytest.importorskip("torch")

from nemaq.models.fusion import GatedFusion


def test_residual_fusion_starts_near_trunk():
    torch.manual_seed(0)
    f = GatedFusion({"bypass": 8, "q": 8, "geo": 8}, 16, mode="residual")
    outs = {n: torch.randn(5, 8) for n in ("bypass", "q", "geo")}
    fused = f(outs)
    trunk = f.proj["bypass"](outs["bypass"])
    g0 = torch.sigmoid(torch.tensor(-2.0))
    expected = trunk + g0 * f.proj["q"](outs["q"]) + g0 * f.proj["geo"](outs["geo"])
    assert torch.allclose(fused, expected, atol=1e-5)
    # residual contribution starts small relative to trunk
    assert float(f.last_gates.mean()) == pytest.approx(float(g0), abs=1e-4)


def test_softmax_fusion_starts_uniform():
    torch.manual_seed(0)
    f = GatedFusion({"a": 8, "b": 8}, 16, mode="softmax")
    f({n: torch.randn(5, 8) for n in ("a", "b")})
    assert torch.allclose(f.last_gates, torch.full_like(f.last_gates, 0.5))


def test_residual_requires_trunk_branch():
    with pytest.raises(ValueError):
        GatedFusion({"a": 8, "b": 8}, 16, mode="residual", trunk="bypass")


def test_gate_dict_excludes_trunk_in_residual_mode():
    f = GatedFusion({"bypass": 8, "q": 8}, 16, mode="residual")
    f({n: torch.randn(3, 8) for n in ("bypass", "q")})
    assert set(f.gate_dict()) == {"q"}


Writing /content/tests/test_fusion.py


In [36]:
%%writefile /content/tests/test_qoa.py
"""QOA faithfulness harness tests (mechanics, not scientific verdicts)."""
import pytest

torch = pytest.importorskip("torch")
pytest.importorskip("pennylane")
pytest.importorskip("geoopt")

from nemaq.models import build_model
from nemaq.telemetry.qoa import (faithfulness_masking, observable_attribution,
                                 perturbation_test, randomization_check)

N, F_IN, C = 20, 12, 3
CFG = {"hidden": 16, "branch_dim": 8, "fused_dim": 16, "n_qubits": 3,
       "q_depth": 1, "geometry": "euclidean", "quantum_mode": "pqc"}


@pytest.fixture(scope="module")
def setup():
    torch.manual_seed(0)
    x = torch.randn(N, F_IN)
    ei = torch.randint(0, N, (2, 60))
    model = build_model("nemaq", F_IN, C, CFG)
    return model, x, ei


def test_attribution_shape_and_grad(setup):
    model, x, ei = setup
    attr, obs = observable_attribution(model, x, ei)
    assert attr.shape == (N, CFG["n_qubits"])
    assert not attr.requires_grad  # detached result


def test_faithfulness_masking_returns_both_drops(setup):
    model, x, ei = setup
    d = faithfulness_masking(model, x, ei, k=1)
    assert set(d) == {"top", "bottom", "faithful"}


def test_perturbation_reports_flip_rates(setup):
    model, x, ei = setup
    f = perturbation_test(model, x, ei, sigma=0.5, k=1)
    assert 0.0 <= f["top"] <= 1.0 and 0.0 <= f["bottom"] <= 1.0


def test_randomization_check_runs(setup):
    model, x, ei = setup
    r = randomization_check(model, x, ei)
    assert -1.0 <= r["spearman_orig_vs_randomized"] <= 1.0


def test_trunk_only_ablation_builds_and_runs():
    cfg = dict(CFG, geometry="off", quantum_mode="off")
    model = build_model("nemaq", F_IN, C, cfg)
    assert list(model.branches) == ["bypass"]
    out = model(torch.randn(N, F_IN), torch.randint(0, N, (2, 40)))
    assert out.shape == (N, C)


Writing /content/tests/test_qoa.py


In [37]:
import sys, os
sys.path.insert(0, "/content/src")          # for this kernel
os.environ["PYTHONPATH"] = "/content/src"   # for the pytest subprocess
os.chdir("/content")
!python -m pytest tests/ -q


............................                                             [100%]
28 passed in 33.34s


## 5. δ-hyperbolicity of Cora (H1 instrument)

δ ≈ 0 → tree-like (hyperbolic encoder should help). Larger δ → less hierarchy.
Run this for every dataset in the suite to build the stratification table for
`experiments/PREREGISTRATION.md`.


In [38]:
from nemaq.data.loader import load_dataset
from nemaq.data.hyperbolicity import sampled_gromov_delta

data = load_dataset("cora", root="/content/data")[0]
stats = sampled_gromov_delta(data.edge_index, data.num_nodes,
                             n_samples=2000, max_component=1500)
print(f"Cora: nodes={data.num_nodes} edges={data.num_edges}")
print(f"delta_mean={stats['delta_mean']:.3f}  delta_max={stats['delta_max']:.1f}"
      f"  (n={stats['n_samples']} sampled 4-tuples)")


Processing...
Done!


Cora: nodes=2708 edges=10556
delta_mean=0.414  delta_max=2.0  (n=2000 sampled 4-tuples)


## 6. GCN baseline (Gate 2 sanity)

Target: ~81.5% test accuracy on the Cora public split. If this is far off,
fix the recipe before touching any exotic component.


In [39]:
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

cfg = load_config("/content/configs/cora_gcn.yaml")
cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"

ds = load_dataset("cora", root="/content/data")
data = apply_split(ds[0].clone(), mode="public")
metrics = train_run(cfg, data, seed=0, run_dir="/content/experiments/demo/cora_gcn/seed0")
print(metrics)


{'val_acc': 0.8100000619888306, 'test_acc': 0.8210000395774841, 'test_macro_f1': 0.8090445620499372, 'test_min_class_f1': 0.7090301003344481, 'epochs_run': 140, 'param_counts': {'total': 92231}}


## 7. Full NEMA-Q (hyperbolic + PQC + classical bypass)

Demo settings: the PQC forward simulates one 4-qubit circuit per node per epoch,
so epochs are capped at 30 here (~a few minutes on CPU). For real runs (Phase 4+)
restore `epochs: 500, patience: 100` and budget the wall-time.

Watch the telemetry summary: the H3 (no-barren-plateau) check is **relative** —
the `q` branch's circuit-weight gradient variance must stay within 2 orders of
magnitude of the best classical branch's variance (`h3_pass_relative`).


In [40]:
cfg = load_config("/content/configs/cora_nemaq.yaml")
cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
cfg["train"]["epochs"] = 30      # DEMO CAP — remove for real runs
cfg["train"]["patience"] = 30

data = apply_split(ds[0].clone(), mode="public")
metrics_q = train_run(cfg, data, seed=0,
                      run_dir="/content/experiments/demo/cora_nemaq/seed0")
print("NEMA-Q:", metrics_q)

import json
man = json.load(open("/content/experiments/demo/cora_nemaq/seed0/manifest.json"))
print("\nPer-branch gradient telemetry (H3 relative check on 'q'):")
for branch, s in man["telemetry_summary"].items():
    line = f"  {branch:8s} min_grad_var={s['min_grad_var']:.2e}"
    if "min_circuit_grad_var" in s:
        line += (f"  circuit_var={s['min_circuit_grad_var']:.2e}"
                 f"  vs_classical_ref={s['h3_classical_reference_var']:.2e}"
                 f"  H3_pass={s['h3_pass_relative']}")
    print(line)


NEMA-Q: {'val_acc': 0.6980000138282776, 'test_acc': 0.6820000410079956, 'test_macro_f1': 0.6889612599486789, 'test_min_class_f1': 0.5498489425981873, 'epochs_run': 30, 'param_counts': {'total': 216799, 'branch_geo': 100097, 'branch_q': 6080, 'branch_bypass': 95936}, 'learned_curvature': 1.1092768907546997}

Per-branch gradient telemetry (H3 relative check on 'q'):
  geo      min_grad_var=4.19e-10
  q        min_grad_var=9.69e-10  circuit_var=6.98e-09  vs_classical_ref=2.28e-09  H3_pass=True
  bypass   min_grad_var=2.28e-09


## 8. NEMA-C — dequantization control (headline comparison)

Same architecture, same recipe; PQC swapped for a parameter-matched classical
surrogate via one config flag. The paper's quantum claim is **NEMA-Q vs NEMA-C**,
never NEMA-Q vs plain GCN.


In [41]:
cfg_s = load_config("/content/configs/ablations/nemaq_surrogate.yaml")
cfg_s["device"] = cfg["device"]
cfg_s["train"]["epochs"] = 30    # DEMO CAP
cfg_s["train"]["patience"] = 30

data = apply_split(ds[0].clone(), mode="public")
metrics_c = train_run(cfg_s, data, seed=0,
                      run_dir="/content/experiments/demo/nemaq_surrogate/seed0")
print("NEMA-C (surrogate):", metrics_c)
print(f"\nNEMA-Q vs NEMA-C test acc: {metrics_q['test_acc']:.4f} vs {metrics_c['test_acc']:.4f}")
print("(1 seed, 30 epochs = anecdote, not evidence. Protocol: 10 seeds, "
      "paired Wilcoxon via nemaq.analysis.stats — see cell below.)")


NEMA-C (surrogate): {'val_acc': 0.5820000171661377, 'test_acc': 0.5940000414848328, 'test_macro_f1': 0.5251117870404662, 'test_min_class_f1': 0.12244897959183673, 'epochs_run': 30, 'param_counts': {'total': 216797, 'branch_geo': 100097, 'branch_q': 6078, 'branch_bypass': 95936}, 'learned_curvature': 1.0523993968963623}

NEMA-Q vs NEMA-C test acc: 0.6820 vs 0.5940
(1 seed, 30 epochs = anecdote, not evidence. Protocol: 10 seeds, paired Wilcoxon via nemaq.analysis.stats — see cell below.)


## 9. Attribution (H4 instrument)

`leave_branch_out` = ground-truth per-node branch usefulness.
`gate_attribution` = cheap proxy from fusion gates.
H4 asks: does the proxy track the ground truth (Spearman ρ > 0.5)?


In [42]:
from nemaq.models import build_model
from nemaq.telemetry.attribution import h4_correlation
from nemaq.utils.seed import set_seed

set_seed(0)
device = torch.device(cfg["device"])
data_d = apply_split(ds[0].clone(), mode="public").to(device)

model = build_model("nemaq", data_d.num_features, int(data_d.y.max()) + 1,
                    cfg["model"]).to(device)
# quick fit so gates are meaningful
import geoopt, torch.nn.functional as F
opt = geoopt.optim.RiemannianAdam(model.parameters(), lr=0.01)
for _ in range(15):
    model.train(); opt.zero_grad()
    out = model(data_d.x, data_d.edge_index)
    F.cross_entropy(out[data_d.train_mask], data_d.y[data_d.train_mask]).backward()
    opt.step()

rho = h4_correlation(model, data_d.x, data_d.edge_index, mask=data_d.test_mask)
print("Spearman(gate, leave-branch-out) per branch:", rho)
print("H4 threshold: rho > 0.5 (expect noise here — model barely trained)")


Spearman(gate, leave-branch-out) per branch: {'geo': 0.2860576981564377, 'q': 0.15686993075236993}
H4 threshold: rho > 0.5 (expect noise here — model barely trained)


## 10. Circuit diagnostics — expressibility & entanglement

Sim et al. 2019 KL-vs-Haar (lower = more expressive) and Meyer–Wallach Q.
Use this sweep to pick (n_qubits, depth) jointly with the H3 gradient-variance
evidence: expressive enough, but still trainable.


In [43]:
from nemaq.telemetry.expressibility import expressibility_kl, meyer_wallach

for nq, depth in [(4, 1), (4, 2)]:
    kl = expressibility_kl(nq, depth, n_pairs=300)   # 300 pairs = quick demo; use 2000+ for the paper
    mw = meyer_wallach(nq, depth, n_samples=100)
    print(f"n_qubits={nq} depth={depth}:  expressibility_KL={kl:.3f}  Meyer-Wallach_Q={mw:.3f}")


n_qubits=4 depth=1:  expressibility_KL=0.308  Meyer-Wallach_Q=0.825
n_qubits=4 depth=2:  expressibility_KL=0.044  Meyer-Wallach_Q=0.866


## 11. Statistical protocol demo

Once the run matrix exists (10 seeds per config), analysis reads manifests only:


In [44]:
from nemaq.analysis.stats import collect_scores, paired_comparison, holm_bonferroni

scores = collect_scores("/content/experiments/demo")
for key, per_seed in scores.items():
    print(key, "->", {s: round(v, 4) for s, v in per_seed.items()})

print("\nWith >=5 paired seeds, run e.g.:")
print("  paired_comparison(scores[nemaq_key], scores[surrogate_key])")
print("  holm_bonferroni({'H1': p1, 'H2': p2})")

# synthetic demo of the machinery:
a = {s: 0.815 + 0.004 * ((s * 7) % 3) for s in range(10)}
b = {s: 0.805 + 0.004 * ((s * 7) % 3) for s in range(10)}
print("\nsynthetic paired_comparison:", paired_comparison(a, b))


('cora', 'gcn', 'public') -> {0: 0.821}
('cora', 'nemaq:pqc:hyperbolic:residual', 'public') -> {0: 0.682}
('cora', 'nemaq:surrogate:hyperbolic:residual', 'public') -> {0: 0.594}

With >=5 paired seeds, run e.g.:
  paired_comparison(scores[nemaq_key], scores[surrogate_key])
  holm_bonferroni({'H1': p1, 'H2': p2})

synthetic paired_comparison: {'n_seeds': 10, 'mean_a': 0.8186, 'std_a': 0.0035023801430836554, 'mean_b': 0.8086, 'std_b': 0.003502380143083656, 'mean_diff': 0.009999999999999898, 'wilcoxon_p': 0.001953125, 'cohens_d': inf}


## 12. Phase-4 matrix (full epochs, 10 seeds)

The real feasibility run: every hybrid variant + baseline, full training,
paired seeds. ~2–3 h on Colab. Includes `nemaq_softmax_fusion` so the paper
can quantify the residual-trunk fix against the unstable softmax gating.
Mount Drive first if you want the manifests to survive the session.


In [45]:
from nemaq.analysis.stats import collect_scores, paired_comparison, holm_bonferroni

MATRIX = ["configs/cora_gcn.yaml", "configs/cora_gat.yaml", "configs/cora_hgcn.yaml",
          "configs/cora_nemaq.yaml",
          "configs/ablations/nemaq_trunk_only.yaml",
          "configs/ablations/nemaq_surrogate.yaml",
          "configs/ablations/nemaq_frozen_random.yaml",
          "configs/ablations/nemaq_euclidean.yaml",
          "configs/ablations/nemaq_softmax_fusion.yaml"]
N_SEEDS = 10

ds_m = load_dataset("cora", root="/content/data")
for cfg_path in MATRIX:
    cfg_m = load_config(f"/content/{cfg_path}")   # full epochs — no demo caps
    cfg_m["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    name = cfg_path.split("/")[-1].removesuffix(".yaml")
    for seed in range(N_SEEDS):
        import os
        run_dir = f"/content/experiments/runs/{name}/seed{seed}"
        if os.path.exists(f"{run_dir}/manifest.json"):
            continue  # resumable
        data_m = apply_split(ds_m[0].clone(), mode="public")
        m = train_run(cfg_m, data_m, seed, run_dir)
        print(f"{name} seed={seed}: test={m['test_acc']:.4f}")

scores = collect_scores("/content/experiments/runs")
import numpy as np
print()
for key, per_seed in sorted(scores.items()):
    v = np.array(list(per_seed.values()))
    print(f"{key[1]:32s} {v.mean():.4f} +/- {v.std(ddof=1):.4f}  (n={len(v)})")

def key_for(model_key):
    return ("cora", model_key, "public")

if key_for("nemaq:pqc:hyperbolic") in scores and key_for("nemaq:surrogate:hyperbolic") in scores:
    h2 = paired_comparison(scores[key_for("nemaq:pqc:hyperbolic")],
                           scores[key_for("nemaq:surrogate:hyperbolic")])
    print("\nH2 primary (NEMA-Q vs NEMA-C):", h2)


cora_gcn seed=0: test=0.8210
cora_gcn seed=1: test=0.8230
cora_gcn seed=2: test=0.8210
cora_gcn seed=3: test=0.8160
cora_gcn seed=4: test=0.8240
cora_gcn seed=5: test=0.8260
cora_gcn seed=6: test=0.8170
cora_gcn seed=7: test=0.8210
cora_gcn seed=8: test=0.7960
cora_gcn seed=9: test=0.8260
cora_gat seed=0: test=0.8290
cora_gat seed=1: test=0.8210
cora_gat seed=2: test=0.8280
cora_gat seed=3: test=0.8300
cora_gat seed=4: test=0.8290
cora_gat seed=5: test=0.8300
cora_gat seed=6: test=0.8240
cora_gat seed=7: test=0.8260
cora_gat seed=8: test=0.8050
cora_gat seed=9: test=0.8090
cora_hgcn seed=0: test=0.7900
cora_hgcn seed=1: test=0.7760
cora_hgcn seed=2: test=0.7940
cora_hgcn seed=3: test=0.7780
cora_hgcn seed=4: test=0.7920
cora_hgcn seed=5: test=0.7770
cora_hgcn seed=6: test=0.7730
cora_hgcn seed=7: test=0.7990
cora_hgcn seed=8: test=0.7930
cora_hgcn seed=9: test=0.7900
cora_nemaq seed=0: test=0.7100
cora_nemaq seed=1: test=0.6890
cora_nemaq seed=2: test=0.7470
cora_nemaq seed=3: test=0.6

## 13. QOA faithfulness validation (ICEQT'26 referee B)

Quantum Observable Attribution + the three checks the referee asked for:
observable masking (top vs bottom attributed), perturbation flip rates, and
model-randomization sanity check. Run on a trained NEMA-Q (train one first,
or load section 7's demo model). Verdicts here go straight into the Q1 paper.


In [46]:
%%writefile /content/src/nemaq/telemetry/qoa.py
"""Quantum Observable Attribution (QOA) + faithfulness validation.

QOA (the ICEQT'26 paper's core method): per-node grad x input of the
predicted-class logit with respect to the measured quantum observables
(per-qubit Pauli-Z expectations), i.e. which observables drive the decision.

Referee B (ICEQT'26): "QOA is gradient saliency over intermediate
observables, but the paper does not provide faithfulness tests,
randomization checks, perturbation tests." This module supplies exactly
those three checks:

  1. faithfulness_masking  — zeroing the top-attributed observable must hurt
     the predicted logit more than zeroing the bottom-attributed one.
  2. perturbation_test     — noise injected on high-attribution observables
     must change predictions more than on low-attribution ones.
  3. randomization_check   — attributions must decay when the downstream
     model (fusion + head) is re-randomized (Adebayo et al. 2018 sanity check).
"""
import copy

import torch

from scipy.stats import spearmanr


def _observables(model, x):
    """Recompute the PQC observable vector [N, n_qubits] outside the graph."""
    qb = model.branches["q"]
    with torch.no_grad():
        angles = torch.tanh(qb.compress(x)) * torch.pi
        obs = qb.qlayer(angles.cpu()).to(x.device)
    return obs


def _logits_from_obs(model, x, edge_index, obs):
    out_q = model.branches["q"].project(obs)
    return model(x, edge_index, branch_override={"q": out_q})


def observable_attribution(model, x, edge_index):
    """Return (attr [N, n_qubits], obs [N, n_qubits]): grad x input of the
    predicted-class logit w.r.t. each measured observable."""
    model.eval()
    obs = _observables(model, x).requires_grad_(True)
    # enable_grad: callers (masking / perturbation checks) run under
    # @torch.no_grad(), which would otherwise leave sel without a graph.
    with torch.enable_grad():
        logits = _logits_from_obs(model, x, edge_index, obs)
        pred = logits.argmax(dim=-1)
        sel = logits[torch.arange(len(pred), device=logits.device), pred].sum()
        (grad,) = torch.autograd.grad(sel, obs)
    return (grad * obs).detach(), obs.detach()


@torch.no_grad()
def faithfulness_masking(model, x, edge_index, k: int = 1, mask=None) -> dict:
    """Zero top-k vs bottom-k attributed observables per node; report mean
    predicted-logit drop. Faithful attribution => drop_top >> drop_bottom."""
    attr, obs = observable_attribution(model, x, edge_index)
    base = _logits_from_obs(model, x, edge_index, obs)
    pred = base.argmax(dim=-1)
    idx = torch.arange(len(pred), device=base.device)
    order = attr.abs().argsort(dim=1)

    drops = {}
    for name, cols in {"top": order[:, -k:], "bottom": order[:, :k]}.items():
        obs_m = obs.clone().scatter_(1, cols, 0.0)
        masked = _logits_from_obs(model, x, edge_index, obs_m)
        d = base[idx, pred] - masked[idx, pred]
        drops[name] = float(d[mask].mean() if mask is not None else d.mean())
    drops["faithful"] = drops["top"] > drops["bottom"]
    return drops


@torch.no_grad()
def perturbation_test(model, x, edge_index, sigma: float = 0.25,
                      k: int = 1, mask=None, seed: int = 0) -> dict:
    """Gaussian noise on top-k vs bottom-k attributed observables; report
    prediction flip rates. Faithful => flip_top > flip_bottom."""
    attr, obs = observable_attribution(model, x, edge_index)
    base_pred = _logits_from_obs(model, x, edge_index, obs).argmax(dim=-1)
    order = attr.abs().argsort(dim=1)
    g = torch.Generator(device="cpu").manual_seed(seed)

    flips = {}
    for name, cols in {"top": order[:, -k:], "bottom": order[:, :k]}.items():
        noise = torch.randn(cols.shape, generator=g).to(obs.device) * sigma
        obs_p = obs.clone().scatter_add_(1, cols, noise)
        pred = _logits_from_obs(model, x, edge_index, obs_p).argmax(dim=-1)
        f = (pred != base_pred).float()
        flips[name] = float(f[mask].mean() if mask is not None else f.mean())
    flips["faithful"] = flips["top"] > flips["bottom"]
    return flips


def randomization_check(model, x, edge_index, seed: int = 0) -> dict:
    """Re-randomize fusion + head; Spearman correlation between original and
    randomized attributions should collapse toward 0 (attribution depends on
    the learned model, not on architecture artifacts)."""
    attr_orig, _ = observable_attribution(model, x, edge_index)

    # the last forward leaves non-leaf tensors (aux logits / branch outputs)
    # cached on the module; deepcopy chokes on them — clear first.
    if hasattr(model, "last_aux_logits"):
        model.last_aux_logits = {}
    if hasattr(model, "last_branch_outputs"):
        model.last_branch_outputs = {}
    rand = copy.deepcopy(model)
    torch.manual_seed(seed)
    for module in (rand.fusion, rand.head):
        for p in module.parameters():
            if p.dim() > 1:
                torch.nn.init.xavier_uniform_(p)
            else:
                torch.nn.init.normal_(p, std=0.1)
    attr_rand, _ = observable_attribution(rand, x, edge_index)

    rho = spearmanr(attr_orig.flatten().cpu().numpy(),
                    attr_rand.flatten().cpu().numpy()).statistic
    return {"spearman_orig_vs_randomized": float(rho),
            "passes": abs(float(rho)) < 0.5}


Overwriting /content/src/nemaq/telemetry/qoa.py


## 14. Phase-5 datasets + EDA (δ-stratified suite)

Adds **Citeseer**, **Pubmed** (Planetoid, public split) and **Disease** (Chami et al. 2019 HGCN repo; δ ≈ 0 tree — H1 positive control; stratified 30/10/60 ratio split).

EDA per dataset: class balance, degree CCDF (log–log), degree by class, homophily, sampled Gromov δ. Also produces the δ-stratification table that fills `PREREGISTRATION.md` §2 and the δ-vs-homophily design figure.

In [47]:
# Phase-5 configs: citeseer / pubmed / disease x {gcn, gat, hgcn, nemaq, euclidean, frozen_random}
import pathlib
CONFIGS = {
 "configs/citeseer_gcn.yaml": "inherit: base.yaml\ndata: { name: citeseer }\n",
 "configs/citeseer_gat.yaml": "inherit: base.yaml\ndata: { name: citeseer }\nmodel: { name: gat }\n",
 "configs/citeseer_hgcn.yaml": "# Standalone hyperbolic baseline (referee B: same-split hyperbolic GNN).\ninherit: base.yaml\ndata: { name: citeseer }\nmodel: { name: hgcn }\n",
 "configs/citeseer_nemaq.yaml": "inherit: base.yaml\ndata: { name: citeseer }\nmodel: { name: nemaq }\n",
 "configs/pubmed_gcn.yaml": "inherit: base.yaml\ndata: { name: pubmed }\n",
 "configs/pubmed_gat.yaml": "inherit: base.yaml\ndata: { name: pubmed }\nmodel: { name: gat }\n",
 "configs/pubmed_hgcn.yaml": "# Standalone hyperbolic baseline (referee B: same-split hyperbolic GNN).\ninherit: base.yaml\ndata: { name: pubmed }\nmodel: { name: hgcn }\n",
 "configs/pubmed_nemaq.yaml": "inherit: base.yaml\ndata: { name: pubmed }\nmodel: { name: nemaq }\n",
 "configs/disease_gcn.yaml": "# Disease (delta ~= 0 tree): H1 positive control. No public split -> ratio.\ninherit: base.yaml\ndata: { name: disease, split: ratio }\n",
 "configs/disease_gat.yaml": "# Disease (delta ~= 0 tree): H1 positive control. No public split -> ratio.\ninherit: base.yaml\ndata: { name: disease, split: ratio }\nmodel: { name: gat }\n",
 "configs/disease_hgcn.yaml": "# Disease (delta ~= 0 tree): H1 positive control. No public split -> ratio.\ninherit: base.yaml\ndata: { name: disease, split: ratio }\nmodel: { name: hgcn }\n",
 "configs/disease_nemaq.yaml": "# Disease (delta ~= 0 tree): H1 positive control. No public split -> ratio.\ninherit: base.yaml\ndata: { name: disease, split: ratio }\nmodel: { name: nemaq }\n",
 "configs/ablations/citeseer_nemaq_euclidean.yaml": "# NEMA-E geometry control on citeseer (H1: geometry x delta interaction).\ninherit: ../base.yaml\ndata: { name: citeseer }\nmodel: { name: nemaq, geometry: euclidean }\n",
 "configs/ablations/citeseer_nemaq_frozen_random.yaml": "# NEMA-R control on citeseer (H5: frozen-random vs trained PQC).\ninherit: ../base.yaml\ndata: { name: citeseer }\nmodel: { name: nemaq, quantum_mode: frozen_random }\n",
 "configs/ablations/pubmed_nemaq_euclidean.yaml": "# NEMA-E geometry control on pubmed (H1: geometry x delta interaction).\ninherit: ../base.yaml\ndata: { name: pubmed }\nmodel: { name: nemaq, geometry: euclidean }\n",
 "configs/ablations/pubmed_nemaq_frozen_random.yaml": "# NEMA-R control on pubmed (H5: frozen-random vs trained PQC).\ninherit: ../base.yaml\ndata: { name: pubmed }\nmodel: { name: nemaq, quantum_mode: frozen_random }\n",
 "configs/ablations/disease_nemaq_euclidean.yaml": "# NEMA-E geometry control on disease \u2014 H1's highest-leverage comparison\n# (delta ~= 0: hyperbolic advantage predicted largest here).\ninherit: ../base.yaml\ndata: { name: disease, split: ratio }\nmodel: { name: nemaq, geometry: euclidean }\n",
 "configs/ablations/disease_nemaq_frozen_random.yaml": "# NEMA-R control on disease (H5 cross-dataset).\ninherit: ../base.yaml\ndata: { name: disease, split: ratio }\nmodel: { name: nemaq, quantum_mode: frozen_random }\n"
}
for rel, text in CONFIGS.items():
    p = pathlib.Path('/content', rel)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
print(f'{len(CONFIGS)} Phase-5 configs written')


18 Phase-5 configs written


In [48]:
%%writefile /content/src/nemaq/analysis/plotstyle.py
"""Shared publication figure style + dataset display metadata.

Every figure module imports from here so the paper's figures are stylistically
uniform (serif, 300 dpi, consistent class palettes across EDA / results / XAI).
"""
import matplotlib.pyplot as plt

PUB_RC = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 300,
    "savefig.bbox": "tight",
}

CLASS_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
]

# Branch palette (fusion / attribution figures)
BRANCH_COLORS = {"q": "#9467bd", "geo": "#2ca02c", "bypass": "#ff7f0e"}
BRANCH_LABELS = {"q": "Quantum (PQC)", "geo": "Hyperbolic (geo)",
                 "bypass": "Classical bypass (trunk)"}

CLASS_NAMES = {
    "cora": ["Case Based", "Genetic Algorithms", "Neural Networks",
             "Probabilistic Meth.", "Reinforcement L.", "Rule Learning",
             "Theory"],
    "citeseer": ["Agents", "AI", "DB", "IR", "ML", "HCI"],
    "pubmed": ["Diabetes Exp.", "Diabetes T1", "Diabetes T2"],
    "disease": ["Not infected", "Infected"],
}

MODEL_LABELS = {
    "gcn": "GCN",
    "gat": "GAT",
    "mlp": "MLP",
    "hgcn": "HGCN",
    "nemaq:pqc:hyperbolic:residual": "NEMA-Q (full)",
    "nemaq:pqc:euclidean:residual": "NEMA-E (Euclidean)",
    "nemaq:frozen_random:hyperbolic:residual": "NEMA-R (frozen PQC)",
    "nemaq:surrogate:hyperbolic:residual": "NEMA-C (surrogate)",
    "nemaq:off:off:residual": "Trunk-only",
    "nemaq:pqc:hyperbolic:softmax": "NEMA-Q (softmax fusion)",
}


def class_names_for(dataset: str, num_classes: int) -> list[str]:
    names = CLASS_NAMES.get(dataset.lower())
    if names and len(names) == num_classes:
        return names
    return [f"C{c}" for c in range(num_classes)]


def apply_style():
    plt.rcParams.update(PUB_RC)


Writing /content/src/nemaq/analysis/plotstyle.py


In [49]:
%%writefile /content/src/nemaq/analysis/eda.py
"""Exploratory data analysis for the evaluation-matrix datasets.

Produces, per dataset:
  - a stats dict (nodes, edges, features, classes, homophily, degree stats,
    sampled Gromov delta, feature sparsity, split sizes),
  - a 4-panel EDA figure (class balance, degree CCDF, hop-distance sample,
    stat summary),
and across datasets:
  - the delta-stratification table (fills PREREGISTRATION.md section 2) and
  - a delta vs homophily scatter that visualizes the H1 control design.
"""
import json
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
from torch_geometric.utils import homophily, to_undirected

from ..data.hyperbolicity import sampled_gromov_delta
from ..data.loader import apply_split, load_dataset
from .plotstyle import CLASS_COLORS, apply_style, class_names_for


def dataset_stats(data, name: str, delta_samples: int = 5000,
                  seed: int = 0) -> dict:
    ei = to_undirected(data.edge_index)
    n = data.num_nodes
    deg = torch.zeros(n, dtype=torch.long).index_add_(
        0, ei[0], torch.ones(ei.size(1), dtype=torch.long))
    y = data.y
    counts = torch.bincount(y).tolist()
    delta = sampled_gromov_delta(ei, n, n_samples=delta_samples, seed=seed)

    g = nx.Graph()
    g.add_nodes_from(range(n))
    g.add_edges_from(ei.t().tolist())
    lcc = max(nx.connected_components(g), key=len)

    stats = {
        "dataset": name,
        "nodes": n,
        "edges_undirected": ei.size(1) // 2,
        "features": data.num_features,
        "classes": len(counts),
        "class_counts": counts,
        "class_imbalance": max(counts) / max(1, min(counts)),
        "edge_homophily": float(homophily(ei, y, method="edge")),
        "mean_degree": float(deg.float().mean()),
        "max_degree": int(deg.max()),
        "feature_density": float((data.x != 0).float().mean()),
        "largest_cc_frac": len(lcc) / n,
        "delta_mean": delta["delta_mean"],
        "delta_max": delta["delta_max"],
    }
    for split in ("train_mask", "val_mask", "test_mask"):
        if hasattr(data, split) and getattr(data, split) is not None:
            m = getattr(data, split)
            if m.dim() == 1:
                stats[split.replace("_mask", "_size")] = int(m.sum())
    return stats


def eda_figure(data, name: str, stats: dict, save_path: str) -> None:
    apply_style()
    ei = to_undirected(data.edge_index)
    n = data.num_nodes
    deg = torch.zeros(n, dtype=torch.long).index_add_(
        0, ei[0], torch.ones(ei.size(1), dtype=torch.long)).numpy()
    y = data.y.numpy()
    k = stats["classes"]
    names = class_names_for(name, k)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    # (a) class balance
    ax = axes[0]
    counts = stats["class_counts"]
    ax.bar(range(k), counts, color=CLASS_COLORS[:k], alpha=0.85)
    ax.set_xticks(range(k))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Nodes")
    ax.set_title("(a) Class balance")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    # (b) degree CCDF (log-log) — scale-free check motivating hyperbolic geometry
    ax = axes[1]
    ds = np.sort(deg[deg > 0])
    ccdf = 1.0 - np.arange(len(ds)) / len(ds)
    ax.loglog(ds, ccdf, ".", ms=3, color="#1f77b4", alpha=0.7)
    ax.set_xlabel("Degree $k$")
    ax.set_ylabel("$P(K \\geq k)$")
    ax.set_title("(b) Degree CCDF (log–log)")
    ax.grid(True, which="both", linestyle=":", alpha=0.5)

    # (c) per-class mean degree — links structure to class
    ax = axes[2]
    mean_deg = [deg[y == c].mean() if (y == c).sum() else 0 for c in range(k)]
    std_deg = [deg[y == c].std() if (y == c).sum() else 0 for c in range(k)]
    ax.bar(range(k), mean_deg, yerr=std_deg, color=CLASS_COLORS[:k],
           alpha=0.85, capsize=3)
    ax.set_xticks(range(k))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Mean degree ± std")
    ax.set_title("(c) Degree by class")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    # (d) stats panel
    ax = axes[3]
    ax.axis("off")
    lines = [
        f"nodes: {stats['nodes']:,}",
        f"edges: {stats['edges_undirected']:,}",
        f"features: {stats['features']:,}"
        f"  (density {stats['feature_density']:.3f})",
        f"classes: {stats['classes']}"
        f"  (imbalance {stats['class_imbalance']:.1f}x)",
        f"edge homophily: {stats['edge_homophily']:.3f}",
        f"mean degree: {stats['mean_degree']:.2f}"
        f"  (max {stats['max_degree']})",
        f"largest CC: {stats['largest_cc_frac'] * 100:.1f}%",
        f"Gromov delta (sampled): mean {stats['delta_mean']:.3f},"
        f" max {stats['delta_max']:.1f}",
    ]
    for key, label in (("train_size", "train"), ("val_size", "val"),
                       ("test_size", "test")):
        if key in stats:
            lines.append(f"{label} nodes: {stats[key]:,}")
    ax.text(0.02, 0.95, "\n".join(lines), va="top", family="monospace",
            fontsize=10, transform=ax.transAxes)
    ax.set_title("(d) Summary")

    fig.suptitle(f"EDA — {name}", fontsize=14, weight="bold")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)


def delta_stratification_figure(all_stats: list[dict], save_path: str) -> None:
    """delta vs homophily scatter — the H1 stratification design in one plot."""
    apply_style()
    fig, ax = plt.subplots(figsize=(7, 5))
    for s in all_stats:
        ax.scatter(s["delta_mean"], s["edge_homophily"],
                   s=np.sqrt(s["nodes"]) * 2, alpha=0.7, zorder=3)
        ax.annotate(s["dataset"], (s["delta_mean"], s["edge_homophily"]),
                    xytext=(6, 4), textcoords="offset points", fontsize=10)
    ax.set_xlabel("Sampled Gromov $\\delta$ (mean) — lower = more tree-like")
    ax.set_ylabel("Edge homophily")
    ax.set_title("Dataset stratification for H1\n"
                 "(marker size $\\propto \\sqrt{N}$)")
    ax.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)


def delta_table_markdown(all_stats: list[dict]) -> str:
    """Fills PREREGISTRATION.md section 2. Roles from delta terciles."""
    rows = sorted(all_stats, key=lambda s: s["delta_mean"])
    lo = rows[0]["delta_mean"]
    hi = rows[-1]["delta_mean"]
    out = ["| Dataset | delta_mean | Role |", "|---|---|---|"]
    for s in rows:
        if s["delta_mean"] <= lo + (hi - lo) / 3:
            role = "positive control (low delta)"
        elif s["delta_mean"] >= lo + 2 * (hi - lo) / 3:
            role = "negative control (high delta)"
        else:
            role = "benchmark"
        out.append(f"| {s['dataset']} | {s['delta_mean']:.3f} | {role} |")
    return "\n".join(out)


def run_eda(datasets: list[str], out_dir: str = "experiments/figures/eda",
            root: str = "data") -> list[dict]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    all_stats = []
    for name in datasets:
        ds = load_dataset(name, root)
        data = ds[0].clone()
        split = "ratio" if name.lower() == "disease" else "public"
        data = apply_split(data, mode=split)
        stats = dataset_stats(data, name)
        all_stats.append(stats)
        eda_figure(data, name, stats, str(out / f"eda_{name}.pdf"))
        print(f"[EDA] {name}: N={stats['nodes']} delta={stats['delta_mean']:.3f}"
              f" homophily={stats['edge_homophily']:.3f}")
    delta_stratification_figure(all_stats, str(out / "delta_stratification.pdf"))
    with open(out / "eda_stats.json", "w", encoding="utf-8") as f:
        json.dump(all_stats, f, indent=2)
    with open(out / "delta_table.md", "w", encoding="utf-8") as f:
        f.write(delta_table_markdown(all_stats) + "\n")
    return all_stats


Writing /content/src/nemaq/analysis/eda.py


In [50]:
from nemaq.analysis.eda import run_eda, delta_table_markdown

eda_stats = run_eda(["cora", "citeseer", "pubmed", "disease"],
                    out_dir="/content/experiments/figures/eda",
                    root="/content/data")
print()
print(delta_table_markdown(eda_stats))  # -> PREREGISTRATION.md §2


[EDA] cora: N=2708 delta=0.356 homophily=0.810


Processing...
Done!


[EDA] citeseer: N=3327 delta=0.550 homophily=0.736


Processing...
Done!


[EDA] pubmed: N=19717 delta=0.399 homophily=0.802
[EDA] disease: N=1044 delta=0.000 homophily=0.875

| Dataset | delta_mean | Role |
|---|---|---|
| disease | 0.000 | positive control (low delta) |
| cora | 0.356 | benchmark |
| pubmed | 0.399 | negative control (high delta) |
| citeseer | 0.550 | negative control (high delta) |


## 15. Phase-5 evaluation matrix (Citeseer / Pubmed / Disease)

Same protocol as §12: 10 seeds, resumable manifests, paired stats.
**Freeze `PREREGISTRATION.md` (incl. H5) and push the repo BEFORE running this cell** — manifests must carry a real git hash.

Optional H1 power extension (pre-declared in the prereg): rerun the two Cora geometry configs with `N_SEEDS = 40`.

In [ ]:
import os
import torch
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

PHASE5 = (
    [f"configs/{d}_{m}.yaml" for d in ("citeseer", "pubmed", "disease")
     for m in ("gcn", "gat", "hgcn", "nemaq")]
    + [f"configs/ablations/{d}_nemaq_{a}.yaml"
       for d in ("citeseer", "pubmed", "disease")
       for a in ("euclidean", "frozen_random")]
)
N_SEEDS = 10

for cfg_path in PHASE5:
    cfg5 = load_config(f"/content/{cfg_path}")
    cfg5["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    ds5 = load_dataset(cfg5["data"]["name"], root="/content/data")
    name = cfg_path.split("/")[-1].removesuffix(".yaml")
    for seed in range(N_SEEDS):
        run_dir = f"/content/experiments/runs/{name}/seed{seed}"
        if os.path.exists(f"{run_dir}/manifest.json"):
            continue  # resumable
        data5 = apply_split(
            ds5[0].clone(),
            mode=cfg5["data"].get("split", "public"),
            labels_per_class=cfg5["data"].get("labels_per_class", 5),
            split_seed=cfg5["data"].get("split_seed", 0),
        )
        m5 = train_run(cfg5, data5, seed, run_dir)
        print(f"{name} seed={seed}: test={m5['test_acc']:.4f}")


citeseer_gcn seed=0: test=0.7170
citeseer_gcn seed=1: test=0.7170
citeseer_gcn seed=2: test=0.7050
citeseer_gcn seed=3: test=0.7160
citeseer_gcn seed=4: test=0.7160
citeseer_gcn seed=5: test=0.7090
citeseer_gcn seed=6: test=0.7270
citeseer_gcn seed=7: test=0.7180
citeseer_gcn seed=8: test=0.7190
citeseer_gcn seed=9: test=0.7150
citeseer_gat seed=0: test=0.7170
citeseer_gat seed=1: test=0.7030
citeseer_gat seed=2: test=0.7050
citeseer_gat seed=3: test=0.7220
citeseer_gat seed=4: test=0.7140
citeseer_gat seed=5: test=0.7170
citeseer_gat seed=6: test=0.7210
citeseer_gat seed=7: test=0.7280
citeseer_gat seed=8: test=0.7110
citeseer_gat seed=9: test=0.7070
citeseer_hgcn seed=0: test=0.6530
citeseer_hgcn seed=1: test=0.6910
citeseer_hgcn seed=2: test=0.6760
citeseer_hgcn seed=3: test=0.6710
citeseer_hgcn seed=4: test=0.6740
citeseer_hgcn seed=5: test=0.6560
citeseer_hgcn seed=6: test=0.6240
citeseer_hgcn seed=7: test=0.6710
citeseer_hgcn seed=8: test=0.6780
citeseer_hgcn seed=9: test=0.6760


In [ ]:
import json, glob
for f in sorted(glob.glob("/content/experiments/runs/citeseer_*/seed*/manifest.json")):
    m = json.load(open(f))
    met = m["metrics"]
    print(f.split("/")[-3], f"seed={m['seed']}",
          f"test={met['test_acc']:.3f} val={met['val_acc']:.3f}",
          f"minF1={met['test_min_class_f1']:.3f}",
          f"macroF1={met['test_macro_f1']:.3f}",
          f"epochs={met['epochs_run']}",
          f"curv={met.get('learned_curvature', '-')}")


In [ ]:
%%writefile /content/configs/ablations/citeseer_nemaq_trunk_only.yaml
# Diagnostic: bypass trunk + fusion + head only (geometry off, quantum off).
inherit: ../base.yaml
data: { name: citeseer }
model: { name: nemaq, geometry: off, quantum_mode: off }


In [ ]:
import pathlib
for ds, extra in [("citeseer", ""), ("pubmed", ""),
                  ("disease", ", split: ratio")]:
    p = pathlib.Path(f"/content/configs/ablations/{ds}_nemaq_trunk_only.yaml")
    p.write_text(
        f"# Diagnostic: bypass trunk + fusion + head only (geometry off, quantum off).\n"
        f"inherit: ../base.yaml\n"
        f"data: {{ name: {ds}{extra} }}\n"
        f"model: {{ name: nemaq, geometry: \"off\", quantum_mode: \"off\" }}\n")
    print("wrote", p)


In [ ]:
import os, torch
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

cfg_t = load_config("/content/configs/ablations/citeseer_nemaq_trunk_only.yaml")
cfg_t["device"] = "cuda" if torch.cuda.is_available() else "cpu"
ds_t = load_dataset("citeseer", root="/content/data")
for seed in range(10):
    run_dir = f"/content/experiments/runs/citeseer_nemaq_trunk_only/seed{seed}"
    if os.path.exists(f"{run_dir}/manifest.json"):
        continue
    data_t = apply_split(ds_t[0].clone(), mode="public")
    m = train_run(cfg_t, data_t, seed, run_dir)
    print(f"trunk_only seed={seed}: test={m['test_acc']:.4f} "
          f"epochs={m['epochs_run']} minF1={m['test_min_class_f1']:.3f}")


In [ ]:
import json, glob
for f in sorted(glob.glob("/content/experiments/runs/citeseer_nemaq/seed*/manifest.json"))[:3]:
    m = json.load(open(f))
    print(f"seed {m['seed']}:")
    print(json.dumps(m["telemetry_summary"], indent=2)[:1500])


In [ ]:
import os, pathlib, itertools
import numpy as np
import torch
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

ds_s = load_dataset("citeseer", root="/content/data")
base = load_config("/content/configs/citeseer_nemaq.yaml")
base["device"] = "cuda" if torch.cuda.is_available() else "cpu"

def run_combo(aux, seed, out_root, patience=100):
    cfg = {**base,
           "model": {**base["model"], "gate_bias_init": -4.0},
           "train": {**base["train"], "lr": 0.005, "aux_weight": aux,
                     "patience": patience}}
    data = apply_split(ds_s[0].clone(), mode="public")
    run_dir = f"{out_root}/citeseer_aux{aux}/seed{seed}"
    return train_run(cfg, data, seed, run_dir)

# ── 1. edge probe: does val keep climbing past aux=1.2? ─────────────────────
val_by_aux = {1.2: 0.6613}   # from sweep2
for aux in (1.5, 2.0):
    vals = [run_combo(aux, s, "/content/experiments/sweep3")["val_acc"]
            for s in range(3)]
    val_by_aux[aux] = sum(vals) / len(vals)
    print(f"probe aux={aux}: mean val={val_by_aux[aux]:.4f}")

best_aux = max(val_by_aux, key=val_by_aux.get)
print(f"\nLOCKED recipe: aux={best_aux} gate_bias=-4.0 lr=0.005 patience=100 "
      f"(val={val_by_aux[best_aux]:.4f})")

# ── 2. one-shot test read: 10 fresh seeds with the locked recipe ────────────
accs, minf1s = [], []
for seed in range(10):
    run_dir = f"/content/experiments/runs/citeseer_nemaq_tuned/seed{seed}"
    if os.path.exists(f"{run_dir}/manifest.json"):
        continue
    m = run_combo(best_aux, seed, "/content/experiments/runs_tmp")
    # run_combo writes under runs_tmp; rewrite into the canonical location
    import shutil
    os.makedirs(os.path.dirname(run_dir), exist_ok=True)
    shutil.move(f"/content/experiments/runs_tmp/citeseer_aux{best_aux}/seed{seed}",
                run_dir)
    accs.append(m["test_acc"])
    minf1s.append(m["test_min_class_f1"])
    print(f"seed={seed}: test={m['test_acc']:.4f} "
          f"minF1={m['test_min_class_f1']:.3f} epochs={m['epochs_run']}")

a = np.array(accs)
print(f"\nciteseer_nemaq TUNED: {a.mean():.4f} +/- {a.std(ddof=1):.4f} (n={len(a)})")
print(f"min class F1 range: {min(minf1s):.3f} – {max(minf1s):.3f}")
print("reference: untuned nemaq 0.5075 | trunk-only 0.6457 | GCN 0.7168")

# ── 3. persist the locked recipe into the config ────────────────────────────
pathlib.Path("/content/configs/citeseer_nemaq.yaml").write_text(
f"""# Tuned on validation only (sweeps 1-3): aux={best_aux} rescues the
# gradient-starved branches; gate_bias -4 mutes the loud q branch at init.
inherit: base.yaml
data: {{ name: citeseer }}
model: {{ name: nemaq, gate_bias_init: -4.0 }}
train: {{ lr: 0.005, aux_weight: {best_aux}, patience: 100 }}
""")
print("\nconfigs/citeseer_nemaq.yaml updated with locked recipe")


In [ ]:
import os, pathlib
import numpy as np
import torch
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

# persist tuned variant configs
for variant, mod in [("euclidean", "geometry: euclidean"),
                     ("frozen_random", "quantum_mode: frozen_random")]:
    pathlib.Path(f"/content/configs/ablations/citeseer_nemaq_{variant}.yaml").write_text(
f"""# Shares the citeseer tuned recipe (selected on validation, sweeps 1-3).
inherit: ../base.yaml
data: {{ name: citeseer }}
model: {{ name: nemaq, {mod}, gate_bias_init: -4.0 }}
train: {{ lr: 0.005, aux_weight: 2.0, patience: 100 }}
""")

ds_v = load_dataset("citeseer", root="/content/data")
for variant in ("euclidean", "frozen_random"):
    cfg_v = load_config(f"/content/configs/ablations/citeseer_nemaq_{variant}.yaml")
    cfg_v["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    accs = []
    for seed in range(10):
        run_dir = f"/content/experiments/runs/citeseer_nemaq_{variant}_tuned/seed{seed}"
        if os.path.exists(f"{run_dir}/manifest.json"):
            continue
        data_v = apply_split(ds_v[0].clone(), mode="public")
        m = train_run(cfg_v, data_v, seed, run_dir)
        accs.append(m["test_acc"])
        print(f"{variant} seed={seed}: test={m['test_acc']:.4f}")
    a = np.array(accs)
    print(f"\nciteseer_{variant} TUNED: {a.mean():.4f} +/- {a.std(ddof=1):.4f}\n")


In [ ]:
import numpy as np
import torch
import networkx as nx
from torch_geometric.utils import to_undirected
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

print("== dataset-level probes ==")
for name in ("cora", "citeseer", "pubmed"):
    data = load_dataset(name, root="/content/data")[0]
    x, ei = data.x, to_undirected(data.edge_index)
    n = data.num_nodes
    # H-a: fragmentation
    g = nx.Graph(); g.add_nodes_from(range(n)); g.add_edges_from(ei.t().tolist())
    comps = list(nx.connected_components(g))
    lcc = max(comps, key=len)
    iso = sum(1 for c in comps if len(c) == 1)
    # H-b: angle collapse through a fresh compress layer (seeded)
    torch.manual_seed(0)
    compress = torch.nn.Linear(x.size(1), 4)
    angles = torch.tanh(compress(x)) * torch.pi
    print(f"{name:9s} LCC={len(lcc)/n:.2%} isolated={iso} "
          f"row_nnz_median={int((x != 0).sum(1).float().median())} "
          f"angle_std={angles.std(0).mean():.4f} "
          f"angle_range={angles.min():.3f}..{angles.max():.3f}")

print("\n== per-node evidence on a collapsed citeseer model (untuned recipe) ==")
cfg_d = load_config("/content/configs/base.yaml")
cfg_d["data"]["name"] = "citeseer"
cfg_d["model"]["name"] = "nemaq"
cfg_d["device"] = "cuda" if torch.cuda.is_available() else "cpu"
ds_d = load_dataset("citeseer", root="/content/data")
data_d = apply_split(ds_d[0].clone(), mode="public").to(cfg_d["device"])
m, model, data_d = train_run(cfg_d, data_d, 2, "/content/experiments/diag/citeseer_untuned/seed2",
                             return_model=True)
print(f"collapsed model test={m['test_acc']:.4f}")

model.eval()
with torch.no_grad():
    logits = model(data_d.x, data_d.edge_index)
    gates = model.fusion.gate_dict()                 # per-node gate per branch
    q_out = model.branches["q"](data_d.x, data_d.edge_index)

correct = (logits.argmax(1) == data_d.y).float()
deg = torch.zeros(data_d.num_nodes, device=correct.device).index_add_(
    0, data_d.edge_index[0], torch.ones(data_d.edge_index.size(1), device=correct.device))
test = data_d.test_mask
iso_mask = (deg == 0) & test
con_mask = (deg > 0) & test
print(f"test acc isolated nodes: {correct[iso_mask].mean():.4f} (n={int(iso_mask.sum())})")
print(f"test acc connected nodes: {correct[con_mask].mean():.4f} (n={int(con_mask.sum())})")
print(f"q branch output per-node std (info content): {q_out.std(0).mean():.5f}")
for bname, g in gates.items():
    print(f"gate[{bname}]: mean={g.mean():.3f} std={g.std():.3f}")


In [ ]:
import os, pathlib
import numpy as np
import torch
from scipy import stats as sps
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

# ── tuned variant configs (same recipe as citeseer_nemaq, selected on val) ──
for variant, mod in [("euclidean", "geometry: euclidean"),
                     ("frozen_random", "quantum_mode: frozen_random")]:
    pathlib.Path(f"/content/configs/ablations/citeseer_nemaq_{variant}.yaml").write_text(
f"""# Shares the citeseer tuned recipe (selected on validation, sweeps 1-3).
inherit: ../base.yaml
data: {{ name: citeseer }}
model: {{ name: nemaq, {mod}, gate_bias_init: -4.0 }}
train: {{ lr: 0.005, aux_weight: 2.0, patience: 100 }}
""")
print("tuned variant configs written")

# ── run both variants, 10 seeds, resumable ──────────────────────────────────
ds_v = load_dataset("citeseer", root="/content/data")
scores = {}
for variant in ("euclidean", "frozen_random"):
    cfg_v = load_config(f"/content/configs/ablations/citeseer_nemaq_{variant}.yaml")
    cfg_v["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    per_seed = {}
    for seed in range(10):
        run_dir = f"/content/experiments/runs/citeseer_nemaq_{variant}_tuned/seed{seed}"
        if os.path.exists(f"{run_dir}/manifest.json"):
            import json
            per_seed[seed] = json.load(open(f"{run_dir}/manifest.json"))["metrics"]["test_acc"]
            continue
        data_v = apply_split(ds_v[0].clone(), mode="public")
        m = train_run(cfg_v, data_v, seed, run_dir)
        per_seed[seed] = m["test_acc"]
        print(f"{variant} seed={seed}: test={m['test_acc']:.4f} "
              f"minF1={m['test_min_class_f1']:.3f}")
    scores[variant] = per_seed
    a = np.array(list(per_seed.values()))
    print(f"\nciteseer_{variant} TUNED: {a.mean():.4f} +/- {a.std(ddof=1):.4f}\n")

# ── paired stats vs the tuned trained model ─────────────────────────────────
import json
trained = {}
for seed in range(10):
    p = f"/content/experiments/runs/citeseer_nemaq_tuned/seed{seed}/manifest.json"
    if os.path.exists(p):
        trained[seed] = json.load(open(p))["metrics"]["test_acc"]

def paired(a_dict, b_dict, label):
    seeds = sorted(set(a_dict) & set(b_dict))
    a = np.array([a_dict[s] for s in seeds])
    b = np.array([b_dict[s] for s in seeds])
    diff = a - b
    p = sps.wilcoxon(a, b).pvalue if not np.allclose(diff, 0) else 1.0
    d = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else float("inf")
    wins = int((diff > 0).sum())
    print(f"{label}: mean diff={diff.mean()*100:+.2f} pts "
          f"(wins {wins}/{len(seeds)}), Wilcoxon p={p:.4f}, d={d:.2f}")

print("== paired comparisons (same seeds, tuned recipe) ==")
paired(scores["frozen_random"], trained, "H5  frozen vs trained     ")
paired(trained, scores["euclidean"], "H1  hyperbolic vs euclidean")


In [ ]:
import itertools, os
import numpy as np
import torch
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

ds_d = load_dataset("disease", root="/content/data")
best = {}
for mname in ("gcn", "gat", "hgcn"):
    base_d = load_config(f"/content/configs/disease_{mname}.yaml")
    base_d["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    results = {}
    for lr, wd, dr in itertools.product([0.01, 0.005, 0.05],
                                        [5e-4, 5e-5], [0.5, 0.2]):
        vals = []
        for seed in range(3):
            cfg_d = {**base_d,
                     "model": {**base_d["model"], "dropout": dr},
                     "train": {**base_d["train"], "lr": lr,
                               "weight_decay": wd, "patience": 200}}
            data_d = apply_split(ds_d[0].clone(), mode="ratio")
            run_dir = f"/content/experiments/sweep_disease/{mname}_lr{lr}_wd{wd}_dr{dr}/seed{seed}"
            m = train_run(cfg_d, data_d, seed, run_dir)
            vals.append(m["val_acc"])
        results[(lr, wd, dr)] = sum(vals) / len(vals)
    b = max(results, key=results.get)
    best[mname] = b
    print(f"{mname}: best lr={b[0]} wd={b[1]} dropout={b[2]} val={results[b]:.4f}")
print("\nbest per model:", best)


In [ ]:
import itertools, os, pathlib, json
import numpy as np
import torch
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

ds_d = load_dataset("disease", root="/content/data")
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def fnum(x):          # YAML-safe float literal (never bare scientific notation)
    return f"{x:.10f}".rstrip("0").rstrip(".")

# ── 1. tuned baseline configs (decimal floats) + 10-seed test read ──────────
LOCKED_BASE = {"gcn": (0.05, 0.0005, 0.5),
               "gat": (0.05, 0.00005, 0.5),
               "hgcn": (0.05, 0.00005, 0.2)}
for mname, (lr, wd, dr) in LOCKED_BASE.items():
    pathlib.Path(f"/content/configs/disease_{mname}.yaml").write_text(
f"""# Disease (delta ~= 0 tree): H1 positive control. Tuned on validation only
# (12-combo sweep; default lr=0.01 left all baselines at majority class 0.7974).
inherit: base.yaml
data: {{ name: disease, split: ratio }}
model: {{ name: {mname}, dropout: {fnum(dr)} }}
train: {{ lr: {fnum(lr)}, weight_decay: {fnum(wd)}, patience: 200 }}
""")
    cfg_b = load_config(f"/content/configs/disease_{mname}.yaml")
    assert isinstance(cfg_b["train"]["weight_decay"], float), mname  # guard
    cfg_b["device"] = DEV
    accs = []
    for seed in range(10):
        run_dir = f"/content/experiments/runs/disease_{mname}_tuned/seed{seed}"
        if os.path.exists(f"{run_dir}/manifest.json"):
            accs.append(json.load(open(f"{run_dir}/manifest.json"))["metrics"]["test_acc"])
            continue
        data_b = apply_split(ds_d[0].clone(), mode="ratio")
        m = train_run(cfg_b, data_b, seed, run_dir)
        accs.append(m["test_acc"])
    a = np.array(accs)
    print(f"disease_{mname} TUNED: {a.mean():.4f} +/- {a.std(ddof=1):.4f}")

# ── 2. NEMA-Q disease sweep (validation only) ───────────────────────────────
base_n = load_config("/content/configs/disease_nemaq.yaml")
base_n["device"] = DEV
results = {}
for lr, aux, gb in itertools.product([0.05, 0.01], [0.3, 1.0], [-2.0, -4.0]):
    vals = []
    for seed in range(3):
        cfg_n = {**base_n,
                 "model": {**base_n["model"], "gate_bias_init": gb},
                 "train": {**base_n["train"], "lr": lr, "aux_weight": aux,
                           "patience": 200}}
        data_n = apply_split(ds_d[0].clone(), mode="ratio")
        run_dir = f"/content/experiments/sweep_disease/nemaq_lr{lr}_aux{aux}_gb{gb}/seed{seed}"
        m = train_run(cfg_n, data_n, seed, run_dir)
        vals.append(m["val_acc"])
    results[(lr, aux, gb)] = sum(vals) / len(vals)
    print(f"nemaq lr={lr} aux={aux} gb={gb}: mean val={results[(lr, aux, gb)]:.4f}")
lr, aux, gb = max(results, key=results.get)
print(f"\nnemaq LOCKED: lr={lr} aux={aux} gate_bias={gb} "
      f"(val={results[(lr, aux, gb)]:.4f})")

# ── 3. NEMA-Q tuned test read ───────────────────────────────────────────────
pathlib.Path("/content/configs/disease_nemaq.yaml").write_text(
f"""# Disease: tuned on validation only (8-combo sweep, equal budget).
inherit: base.yaml
data: {{ name: disease, split: ratio }}
model: {{ name: nemaq, gate_bias_init: {fnum(gb)} }}
train: {{ lr: {fnum(lr)}, aux_weight: {fnum(aux)}, patience: 200 }}
""")
cfg_f = load_config("/content/configs/disease_nemaq.yaml")
cfg_f["device"] = DEV
accs = []
for seed in range(10):
    run_dir = f"/content/experiments/runs/disease_nemaq_tuned/seed{seed}"
    if os.path.exists(f"{run_dir}/manifest.json"):
        accs.append(json.load(open(f"{run_dir}/manifest.json"))["metrics"]["test_acc"])
        continue
    data_f = apply_split(ds_d[0].clone(), mode="ratio")
    m = train_run(cfg_f, data_f, seed, run_dir)
    accs.append(m["test_acc"])
    print(f"nemaq seed={seed}: test={m['test_acc']:.4f}")
a = np.array(accs)
print(f"\ndisease_nemaq TUNED: {a.mean():.4f} +/- {a.std(ddof=1):.4f}")
print("reference (untuned): nemaq 0.8568 | baselines 0.7974 (degenerate)")


In [ ]:
import os, pathlib, json
import numpy as np
import torch
from scipy import stats as sps
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

ds_d = load_dataset("disease", root="/content/data")
DEV = "cuda" if torch.cuda.is_available() else "cpu"

for variant, mod in [("euclidean", "geometry: euclidean"),
                     ("frozen_random", "quantum_mode: frozen_random")]:
    pathlib.Path(f"/content/configs/ablations/disease_nemaq_{variant}.yaml").write_text(
f"""# Shares the disease tuned recipe (selected on validation).
inherit: ../base.yaml
data: {{ name: disease, split: ratio }}
model: {{ name: nemaq, {mod}, gate_bias_init: -2.0 }}
train: {{ lr: 0.05, aux_weight: 0.3, patience: 200 }}
""")

scores = {}
for variant in ("euclidean", "frozen_random"):
    cfg_v = load_config(f"/content/configs/ablations/disease_nemaq_{variant}.yaml")
    cfg_v["device"] = DEV
    per_seed = {}
    for seed in range(10):
        run_dir = f"/content/experiments/runs/disease_nemaq_{variant}_tuned/seed{seed}"
        if os.path.exists(f"{run_dir}/manifest.json"):
            per_seed[seed] = json.load(open(f"{run_dir}/manifest.json"))["metrics"]["test_acc"]
            continue
        data_v = apply_split(ds_d[0].clone(), mode="ratio")
        m = train_run(cfg_v, data_v, seed, run_dir)
        per_seed[seed] = m["test_acc"]
        print(f"{variant} seed={seed}: test={m['test_acc']:.4f}")
    scores[variant] = per_seed
    a = np.array(list(per_seed.values()))
    print(f"disease_{variant} TUNED: {a.mean():.4f} +/- {a.std(ddof=1):.4f}\n")

trained = {s: json.load(open(f"/content/experiments/runs/disease_nemaq_tuned/seed{s}/manifest.json"))["metrics"]["test_acc"]
           for s in range(10)}

def paired(a_d, b_d, label):
    seeds = sorted(set(a_d) & set(b_d))
    a = np.array([a_d[s] for s in seeds]); b = np.array([b_d[s] for s in seeds])
    diff = a - b
    p = sps.wilcoxon(a, b).pvalue if not np.allclose(diff, 0) else 1.0
    print(f"{label}: diff={diff.mean()*100:+.2f} pts (wins {int((diff>0).sum())}/{len(seeds)}), "
          f"Wilcoxon p={p:.4f}")

print("== paired (tuned recipe, same seeds) ==")
paired(scores["frozen_random"], trained, "H5  frozen vs trained      ")
paired(trained, scores["euclidean"], "H1  hyperbolic vs euclidean")


## 16. Results figures (forest, waterfall, stability, paired diffs, H1 geometry×δ)

All figures read manifests only. The **geometry×δ** figure is the registered fix for the underpowered single-dataset hyperbolic-vs-Euclidean comparison (Cora Phase 4: paired d = 0.44, p = 0.20 at n = 10): per-dataset one-sided Wilcoxon pooled with Stouffer's Z, plus the Spearman(δ, gain) trend H1 predicts.

In [ ]:
%%writefile /content/src/nemaq/analysis/figures.py
"""Paper results figures. All consume run manifests only (design rule #1).

  forest_plot           — per-model mean test acc with 95% CI, one dataset.
  component_waterfall   — the component-accounting story: GCN -> scaffold ->
                          +frozen PQC -> trained PQC, as cumulative deficits.
  paired_diff_plot      — seed-paired slope chart for any two models.
  stability_plot        — across-seed std per model (fusion stability claim).
  geometry_delta_plot   — H1 headline: paired (hyperbolic - Euclidean) gain
                          vs dataset delta, with Spearman trend.
"""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats as sps

from .plotstyle import MODEL_LABELS, apply_style
from .stats import collect_scores


def _label(key: str) -> str:
    return MODEL_LABELS.get(key, key)


def _series(grouped, dataset, model_key, split="public"):
    for (ds, mk, sp), scores in grouped.items():
        if ds == dataset and mk == model_key and sp == split:
            return scores
    return None


def _mean_ci(vals: np.ndarray, conf: float = 0.95) -> tuple[float, float]:
    m = vals.mean()
    if len(vals) < 2:
        return m, 0.0
    half = sps.t.ppf(0.5 + conf / 2, len(vals) - 1) * vals.std(ddof=1) / np.sqrt(len(vals))
    return m, half


def forest_plot(runs_root: str, dataset: str, save_path: str,
                split: str = "public") -> None:
    grouped = collect_scores(runs_root)
    rows = []
    for (ds, mk, sp), scores in sorted(grouped.items()):
        if ds != dataset or sp != split:
            continue
        vals = np.array(list(scores.values()))
        m, half = _mean_ci(vals)
        rows.append((_label(mk), m, half, len(vals)))
    if not rows:
        raise ValueError(f"No manifests for dataset={dataset} split={split}")
    rows.sort(key=lambda r: r[1])

    apply_style()
    fig, ax = plt.subplots(figsize=(7, 0.5 * len(rows) + 2))
    ys = np.arange(len(rows))
    for y, (name, m, half, n) in zip(ys, rows):
        ax.errorbar(m, y, xerr=half, fmt="o", color="#1f77b4",
                    ecolor="#1f77b4", capsize=4, ms=6)
        ax.text(m + half + 0.004, y, f"{m:.3f} ± {half:.3f} (n={n})",
                va="center", fontsize=8)
    ax.set_yticks(ys)
    ax.set_yticklabels([r[0] for r in rows], fontsize=10)
    ax.set_xlabel("Test accuracy (mean, 95% CI)")
    ax.set_title(f"{dataset} — model comparison ({split} split)")
    ax.grid(True, axis="x", linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)


WATERFALL_STAGES = [
    ("gcn", "GCN\n(reference)"),
    ("nemaq:off:off:residual", "+ scaffold\n(trunk-only)"),
    ("nemaq:frozen_random:hyperbolic:residual", "+ frozen\nrandom PQC"),
    ("nemaq:pqc:hyperbolic:residual", "+ trained\nPQC"),
]


def component_waterfall(runs_root: str, dataset: str, save_path: str,
                        split: str = "public") -> None:
    """Cumulative accuracy at each architectural stage, with the per-stage
    paired deficit annotated (the paper's component-accounting figure)."""
    grouped = collect_scores(runs_root)
    series = []
    for key, label in WATERFALL_STAGES:
        s = _series(grouped, dataset, key, split)
        if s is None:
            raise ValueError(f"Missing runs for {key} on {dataset}")
        series.append((label, s))

    apply_style()
    fig, ax = plt.subplots(figsize=(8, 5))
    xs = np.arange(len(series))
    means, halves = [], []
    for _, s in series:
        m, h = _mean_ci(np.array(list(s.values())))
        means.append(m)
        halves.append(h)
    colors = ["#7f7f7f", "#ff7f0e", "#2ca02c", "#9467bd"]
    ax.bar(xs, means, yerr=halves, color=colors, alpha=0.85, capsize=5)

    # paired per-stage deltas (same seeds)
    for i in range(1, len(series)):
        prev, cur = series[i - 1][1], series[i][1]
        seeds = sorted(set(prev) & set(cur))
        diff = np.array([cur[s] - prev[s] for s in seeds])
        p = sps.wilcoxon(diff).pvalue if not np.allclose(diff, 0) else 1.0
        ax.annotate(
            f"{diff.mean() * 100:+.1f} pts\np={p:.3g}",
            xy=((xs[i - 1] + xs[i]) / 2, max(means[i - 1], means[i]) + 0.012),
            ha="center", fontsize=9, color="#d62728")
    ax.set_xticks(xs)
    ax.set_xticklabels([lbl for lbl, _ in series], fontsize=10)
    ax.set_ylabel("Test accuracy")
    lo = min(means) - 0.05
    ax.set_ylim(lo, max(means) + 0.05)
    ax.set_title(f"{dataset} — component accounting\n"
                 "(paired per-stage deficits, Wilcoxon p)")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)


def paired_diff_plot(runs_root: str, dataset: str, model_a: str, model_b: str,
                     save_path: str, split: str = "public") -> None:
    """Slope chart: per-seed accuracy of two models + paired diff histogram."""
    grouped = collect_scores(runs_root)
    sa = _series(grouped, dataset, model_a, split)
    sb = _series(grouped, dataset, model_b, split)
    if sa is None or sb is None:
        raise ValueError(f"Missing runs for {model_a} or {model_b} on {dataset}")
    seeds = sorted(set(sa) & set(sb))
    a = np.array([sa[s] for s in seeds])
    b = np.array([sb[s] for s in seeds])
    diff = a - b
    p = sps.wilcoxon(a, b).pvalue if not np.allclose(diff, 0) else 1.0

    apply_style()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5),
                             gridspec_kw={"width_ratios": [1.2, 1]})
    ax = axes[0]
    for i, s in enumerate(seeds):
        ax.plot([0, 1], [b[i], a[i]], "-o", ms=4, alpha=0.6,
                color="#2ca02c" if diff[i] > 0 else "#d62728")
    ax.set_xticks([0, 1])
    ax.set_xticklabels([_label(model_b), _label(model_a)], fontsize=10)
    ax.set_ylabel("Test accuracy")
    ax.set_title(f"{dataset}: per-seed paired comparison")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    ax = axes[1]
    ax.hist(diff, bins=max(5, len(seeds) // 2), color="#1f77b4",
            alpha=0.8, edgecolor="black", linewidth=0.4)
    ax.axvline(0, color="black", lw=1)
    ax.axvline(diff.mean(), color="#d62728", linestyle="--", lw=1.5,
               label=f"mean {diff.mean() * 100:+.1f} pts")
    ax.set_xlabel(f"{_label(model_a)} − {_label(model_b)}")
    ax.set_title(f"Paired diff (n={len(seeds)}, Wilcoxon p={p:.3g})")
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)


def stability_plot(runs_root: str, dataset: str, save_path: str,
                   split: str = "public") -> None:
    """Across-seed std per model — the fusion-stability claim in one figure."""
    grouped = collect_scores(runs_root)
    rows = []
    for (ds, mk, sp), scores in sorted(grouped.items()):
        if ds != dataset or sp != split:
            continue
        vals = np.array(list(scores.values()))
        rows.append((_label(mk), vals.std(ddof=1), len(vals)))
    rows.sort(key=lambda r: r[1])

    apply_style()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    xs = np.arange(len(rows))
    colors = ["#d62728" if "softmax" in n.lower() or "surrogate" in n.lower()
              else "#1f77b4" for n, _, _ in rows]
    ax.bar(xs, [r[1] for r in rows], color=colors, alpha=0.85)
    ax.set_xticks(xs)
    ax.set_xticklabels([r[0] for r in rows], rotation=30, ha="right",
                       fontsize=9)
    ax.set_ylabel("Across-seed std of test accuracy")
    ax.set_title(f"{dataset} — seed stability by model")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)


def geometry_delta_plot(runs_root: str, delta_map: dict[str, float],
                        save_path: str,
                        hyp_key: str = "nemaq:pqc:hyperbolic:residual",
                        euc_key: str = "nemaq:pqc:euclidean:residual") -> dict:
    """H1 figure: per-dataset paired (hyperbolic - Euclidean) gain vs delta.

    delta_map: {dataset: delta_mean} (from EDA stats json).
    Returns the underlying numbers, including Spearman(delta, gain) and a
    Stouffer-combined p across datasets — the pooled-power answer to the
    single-dataset underpowering of the geometry comparison.
    """
    grouped = collect_scores(runs_root)
    points = []
    for ds, delta in delta_map.items():
        # match whatever split the runs used for this dataset
        sh = se = None
        for (d, mk, sp), scores in grouped.items():
            if d != ds:
                continue
            if mk == hyp_key:
                sh = scores
            elif mk == euc_key:
                se = scores
        if sh is None or se is None:
            continue
        seeds = sorted(set(sh) & set(se))
        if len(seeds) < 5:
            continue
        diff = np.array([sh[s] - se[s] for s in seeds])
        t, p_two = sps.ttest_rel([sh[s] for s in seeds], [se[s] for s in seeds])
        m, half = _mean_ci(diff)
        points.append({"dataset": ds, "delta": delta, "gain": m, "ci": half,
                       "n": len(seeds), "t": float(t), "p": float(p_two)})
    if len(points) < 2:
        raise ValueError("Need geometry-pair runs on >= 2 datasets")

    deltas = np.array([p["delta"] for p in points])
    gains = np.array([p["gain"] for p in points])
    rho, rho_p = sps.spearmanr(deltas, gains)
    # Stouffer: one-sided z per dataset (H1 direction: gain > 0), pooled
    zs = np.array([sps.norm.isf(sps.t.sf(p["t"], p["n"] - 1)) for p in points])
    z_pooled = zs.sum() / np.sqrt(len(zs))
    p_pooled = float(sps.norm.sf(z_pooled))

    apply_style()
    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.axhline(0, color="black", lw=1)
    ax.errorbar(deltas, gains, yerr=[p["ci"] for p in points], fmt="o",
                ms=7, capsize=4, color="#1f77b4")
    for p in points:
        ax.annotate(f"{p['dataset']}\n(n={p['n']})", (p["delta"], p["gain"]),
                    xytext=(8, 6), textcoords="offset points", fontsize=9)
    ax.set_xlabel("Sampled Gromov $\\delta$ (mean)")
    ax.set_ylabel("Paired gain: hyperbolic − Euclidean (test acc)")
    ax.set_title("H1: geometry gain vs hyperbolicity\n"
                 f"Spearman $\\rho$={rho:.2f} (p={rho_p:.3g});  "
                 f"pooled one-sided p={p_pooled:.3g} (Stouffer)")
    ax.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)
    return {"points": points, "spearman_rho": float(rho),
            "spearman_p": float(rho_p), "stouffer_p": p_pooled}


def make_all(runs_root: str, dataset: str, out_dir: str,
             split: str = "public") -> None:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    forest_plot(runs_root, dataset, str(out / f"{dataset}_forest.pdf"), split)
    stability_plot(runs_root, dataset, str(out / f"{dataset}_stability.pdf"), split)
    try:
        component_waterfall(runs_root, dataset,
                            str(out / f"{dataset}_waterfall.pdf"), split)
    except ValueError as e:
        print(f"[figures] waterfall skipped for {dataset}: {e}")
    for a, b, tag in [
        ("nemaq:pqc:hyperbolic:residual", "nemaq:pqc:euclidean:residual",
         "geometry"),
        ("nemaq:frozen_random:hyperbolic:residual",
         "nemaq:pqc:hyperbolic:residual", "frozen_vs_trained"),
    ]:
        try:
            paired_diff_plot(runs_root, dataset, a, b,
                             str(out / f"{dataset}_paired_{tag}.pdf"), split)
        except ValueError as e:
            print(f"[figures] paired {tag} skipped for {dataset}: {e}")


In [ ]:
import json
from nemaq.analysis.figures import make_all, geometry_delta_plot
from nemaq.analysis.stats import geometry_interaction

RUNS, OUT = "/content/experiments/runs", "/content/experiments/figures/results"
for ds in ("cora", "citeseer", "pubmed", "disease"):
    split = "ratio" if ds == "disease" else "public"
    try:
        make_all(RUNS, ds, OUT, split=split)
        print(f"[figures] {ds}: done")
    except ValueError as e:
        print(f"[figures] {ds}: skipped ({e})")

with open("/content/experiments/figures/eda/eda_stats.json") as f:
    delta_map = {s["dataset"]: s["delta_mean"] for s in json.load(f)}
try:
    res = geometry_delta_plot(RUNS, delta_map, f"{OUT}/h1_geometry_delta.pdf")
    print(f"H1 pooled (Stouffer, one-sided): p={res['stouffer_p']:.4g}")
    print(f"H1 trend: Spearman rho={res['spearman_rho']:.2f} "
          f"(p={res['spearman_p']:.3g})")
    print(geometry_interaction(RUNS, delta_map))  # registered test
except ValueError as e:
    print(f"geometry x delta needs the geometry pair on >= 2 datasets: {e}")


## 17. XAI figure suite (IG, QOA, Poincaré disk, fusion decomposition, branch Shapley)

Port of the ICEQT'26 v9 XAI module to the branch architecture, generalized to all datasets:

- **XAI-01** Integrated Gradients on input features (per predicted class)
- **XAI-02** Quantum Observable Attribution heatmap + faithfulness panel (masking / perturbation / randomization — the referee-B harness)
- **XAI-03** Poincaré disk view of the geo branch (2D PCA of tangent output, exp-mapped at the learned curvature; class prototypes)
- **XAI-04** Gradient-attributed per-node quantum contribution r_Q
- **XAI-05** Exact branch Shapley (2^B coalitions — replaces v9 KernelSHAP; fused dims are no longer individually meaningful)

Trains one run per dataset (median-accuracy seed) then renders all figures. Pubmed IG is the slow step — lower `ig_samples` if needed.

In [ ]:
%%writefile /content/src/nemaq/analysis/xai.py
"""XAI figure suite for NEMA-Q (port of the ICEQT'26 v9 XAI module to the
branch architecture, generalized to any dataset).

  xai_integrated_gradients — IG on input features per predicted class.
  xai_qoa_figures          — QOA heatmap (class x observable) + global ranking
                             (uses telemetry.qoa.observable_attribution).
  xai_qoa_faithfulness_fig — masking / perturbation / randomization panel
                             (the referee-B validation harness, visualized).
  xai_poincare             — Poincare-disk view of the geo branch (2D PCA of
                             the tangent output, exp-mapped to the disk).
  xai_fusion_decomposition — gradient-attributed per-node r_Q (quantum
                             contribution ratio) distributions.
  xai_branch_shapley       — exact Shapley values over branches (2^B
                             coalitions via disable_branch) per node/class.

All figures are dataset-generic: class count from labels, display names from
plotstyle.class_names_for.
"""
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from ..telemetry.qoa import (faithfulness_masking, observable_attribution,
                             perturbation_test, randomization_check)
from .plotstyle import (BRANCH_COLORS, BRANCH_LABELS, CLASS_COLORS,
                        apply_style, class_names_for)


def _np(t: torch.Tensor) -> np.ndarray:
    return t.detach().cpu().numpy()


def _device(model) -> torch.device:
    return next(model.parameters()).device


def _class_grid(k: int) -> tuple[int, int]:
    cols = min(4, k)
    return math.ceil(k / cols), cols


def _branch_grads(model, x, edge_index):
    """Forward with every branch output substituted by a requires_grad leaf;
    backward the predicted-class logits. Returns per-branch |grad| L1 per
    node — the gradient-attributed branch sensitivity used for r_Q."""
    model.eval()
    with torch.no_grad():
        outs = {n: b(x, edge_index) for n, b in model.branches.items()}
    leaves = {n: o.detach().requires_grad_(True) for n, o in outs.items()}
    logits = model(x, edge_index, branch_override=leaves)
    pred = logits.argmax(dim=-1)
    sel = logits[torch.arange(len(pred), device=logits.device), pred].sum()
    grads = torch.autograd.grad(sel, list(leaves.values()))
    a = {n: g.abs().sum(dim=1) for n, g in zip(leaves, grads)}
    return a, logits.detach(), pred.detach()


def quantum_contribution_ratio(model, x, edge_index):
    """r_Q(i) = |grad wrt q| / sum over branches — gradient-attributed
    (decision sensitivity, not activation energy). Returns (r_q, logits)."""
    a, logits, _ = _branch_grads(model, x, edge_index)
    total = sum(a.values()) + 1e-8
    return (a["q"] / total if "q" in a else torch.zeros_like(total)), logits


# ── XAI-01: Integrated Gradients on input features ─────────────────────────

def xai_integrated_gradients(model, data, dataset: str, save_path: str,
                             n_steps: int = 32, n_samples: int = 70,
                             top_k: int = 20, seed: int = 42) -> np.ndarray:
    model.eval()
    dev = _device(model)
    x, ei, y = data.x, data.edge_index, data.y
    test_idx = data.test_mask.nonzero(as_tuple=True)[0]
    k = int(y.max()) + 1
    names = class_names_for(dataset, k)

    rng = np.random.default_rng(seed)
    sample = []
    per_class = max(1, n_samples // k)
    for c in range(k):
        idxs = _np(test_idx[(y[test_idx] == c).cpu()])
        if len(idxs):
            sample.extend(rng.choice(idxs, size=min(per_class, len(idxs)),
                                     replace=False).tolist())

    attr, preds = [], []
    alphas = torch.linspace(0, 1, n_steps, device=dev)
    for node in sample:
        with torch.no_grad():
            pred = int(model(x, ei).argmax(dim=-1)[node])
        preds.append(pred)
        grad_sum = torch.zeros(x.size(1), device=dev)
        for alpha in alphas:
            x_mod = (alpha * x).detach()
            row = x_mod[node].clone().requires_grad_(True)
            x_in = x_mod.index_copy(
                0, torch.tensor([node], device=dev), row.unsqueeze(0))
            logit = model(x_in, ei)[node, pred]
            (g,) = torch.autograd.grad(logit, row)
            grad_sum += g
        attr.append(_np(x[node] * grad_sum / n_steps))
    attr = np.array(attr)
    preds = np.array(preds)

    apply_style()
    rows, cols = _class_grid(k)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows),
                             squeeze=False)
    for c in range(k):
        ax = axes[c // cols][c % cols]
        mask = preds == c
        if mask.sum() == 0:
            ax.set_title(f"{names[c]} (no samples)")
            ax.axis("off")
            continue
        mean_abs = np.abs(attr[mask]).mean(axis=0)
        top = np.argsort(mean_abs)[-top_k:][::-1]
        ax.barh(range(len(top)), mean_abs[top],
                color=CLASS_COLORS[c % len(CLASS_COLORS)], alpha=0.85)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels([f"f{i}" for i in top], fontsize=7)
        ax.invert_yaxis()
        ax.set_title(f"{names[c]} (n={int(mask.sum())})", fontsize=10)
        ax.set_xlabel("Mean |IG|", fontsize=9)
        ax.grid(True, axis="x", linestyle=":", alpha=0.6)
    for j in range(k, rows * cols):
        axes[j // cols][j % cols].axis("off")
    fig.suptitle(f"{dataset} — XAI-01 Integrated Gradients: "
                 f"top-{top_k} input features per predicted class",
                 fontsize=13, weight="bold")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)
    return attr


# ── XAI-02: Quantum Observable Attribution figures ──────────────────────────

def xai_qoa_figures(model, data, dataset: str, save_path: str) -> np.ndarray:
    x, ei, y = data.x, data.edge_index, data.y
    test = data.test_mask
    k = int(y.max()) + 1
    names = class_names_for(dataset, k)
    nq = model.branches["q"].n_qubits
    obs_names = [f"$\\langle Z_{i} \\rangle$" for i in range(nq)]

    attr, _ = observable_attribution(model, x, ei)
    with torch.no_grad():
        preds = model(x, ei).argmax(dim=-1)
    attr_t = _np(attr[test].abs())
    preds_t = _np(preds[test])

    heat = np.zeros((k, nq))
    for c in range(k):
        m = preds_t == c
        if m.sum():
            heat[c] = attr_t[m].mean(axis=0)

    apply_style()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5),
                             gridspec_kw={"width_ratios": [1.8, 1]})
    ax = axes[0]
    im = ax.imshow(heat, cmap="YlOrRd", aspect="auto")
    plt.colorbar(im, ax=ax, label="Mean |grad × obs|")
    ax.set_xticks(range(nq))
    ax.set_xticklabels(obs_names, fontsize=10)
    ax.set_yticks(range(k))
    ax.set_yticklabels(names, fontsize=9)
    for c in range(k):
        for q in range(nq):
            ax.text(q, c, f"{heat[c, q]:.2f}", ha="center", va="center",
                    fontsize=8,
                    color="white" if heat[c, q] > heat.max() * 0.6 else "black")
    ax.set_title("QOA: attribution per class × observable")

    ax2 = axes[1]
    glob = attr_t.mean(axis=0)
    rank = np.argsort(glob)[::-1]
    ax2.barh(range(nq), glob[rank], color="#9467bd", alpha=0.85)
    ax2.set_yticks(range(nq))
    ax2.set_yticklabels([obs_names[r] for r in rank], fontsize=10)
    ax2.invert_yaxis()
    ax2.set_xlabel("Global mean |QOA|")
    ax2.set_title("Observable importance")
    ax2.grid(True, axis="x", linestyle=":", alpha=0.6)

    fig.suptitle(f"{dataset} — XAI-02 Quantum Observable Attribution",
                 fontsize=13, weight="bold")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)
    return attr_t


def xai_qoa_faithfulness_fig(model, data, dataset: str, save_path: str,
                             k_obs: int = 1) -> dict:
    """Referee-B validation harness as one panel: masking drops,
    perturbation flip rates, randomization correlation."""
    x, ei = data.x, data.edge_index
    test = data.test_mask
    masking = faithfulness_masking(model, x, ei, k=k_obs, mask=test)
    perturb = perturbation_test(model, x, ei, k=k_obs, mask=test)
    randch = randomization_check(model, x, ei)

    apply_style()
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    ax = axes[0]
    ax.bar([0, 1], [masking["top"], masking["bottom"]],
           color=["#d62728", "#1f77b4"], alpha=0.85)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["mask top-attr", "mask bottom-attr"])
    ax.set_ylabel("Mean predicted-logit drop")
    ax.set_title(f"(a) Masking — faithful: {masking['faithful']}")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    ax = axes[1]
    ax.bar([0, 1], [perturb["top"], perturb["bottom"]],
           color=["#d62728", "#1f77b4"], alpha=0.85)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["perturb top-attr", "perturb bottom-attr"])
    ax.set_ylabel("Prediction flip rate")
    ax.set_title(f"(b) Perturbation — faithful: {perturb['faithful']}")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    ax = axes[2]
    rho = randch["spearman_orig_vs_randomized"]
    ax.bar([0], [abs(rho)], color="#2ca02c" if randch["passes"] else "#d62728",
           alpha=0.85)
    ax.axhline(0.5, color="black", linestyle="--", lw=1.2,
               label="pass threshold (|ρ| < 0.5)")
    ax.set_xticks([0])
    ax.set_xticklabels(["|Spearman ρ|\norig vs randomized"])
    ax.set_ylim(0, 1)
    ax.set_title(f"(c) Randomization — passes: {randch['passes']}")
    ax.legend(fontsize=8)
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    fig.suptitle(f"{dataset} — QOA faithfulness validation "
                 "(masking / perturbation / randomization)",
                 fontsize=13, weight="bold")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)
    return {"masking": masking, "perturbation": perturb,
            "randomization": randch}


# ── XAI-03: Poincare disk topology ──────────────────────────────────────────

def xai_poincare(model, data, dataset: str, save_path: str) -> None:
    if "geo" not in model.branches or not hasattr(model.branches["geo"],
                                                  "manifold"):
        print("[XAI-03] no hyperbolic geo branch — skipped")
        return
    x, ei, y = data.x, data.edge_index, data.y
    test_idx = data.test_mask.nonzero(as_tuple=True)[0]
    k = int(y.max()) + 1
    names = class_names_for(dataset, k)
    geo = model.branches["geo"]

    model.eval()
    with torch.no_grad():
        tangent = geo(x, ei)                      # [N, emb] tangent at origin
        logits = model(x, ei)
    r_q, _ = quantum_contribution_ratio(model, x, ei)

    # 2D PCA of tangent vectors, exp-mapped back onto the (2D) disk at the
    # learned curvature — a faithful "view" of the hyperbolic embedding.
    t_np = _np(tangent)
    t_c = t_np - t_np.mean(axis=0)
    _, _, vt = np.linalg.svd(t_c, full_matrices=False)
    t2 = torch.tensor(t_c @ vt[:2].T, dtype=torch.float32)
    manifold = geo.manifold
    with torch.no_grad():
        disk = _np(manifold.projx(manifold.expmap0(t2.to(tangent.device))))

    ti = _np(test_idx)
    pts = disk[ti]
    true_np = _np(y[test_idx])
    probs = _np(F.softmax(logits[test_idx], dim=1))
    preds_np = probs.argmax(axis=1)
    conf_np = probs.max(axis=1)
    rq_np = _np(r_q[test_idx])

    # class prototypes: tangent-space mean, exp-mapped (Frechet 1st-order)
    protos = {}
    y_np = _np(y)
    for c in range(k):
        m = y_np == c
        if m.sum() < 2:
            continue
        mu = torch.tensor((t_c[m]).mean(axis=0) @ vt[:2].T,
                          dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            protos[c] = _np(manifold.projx(
                manifold.expmap0(mu.to(tangent.device))))[0]

    apply_style()
    fig, axes = plt.subplots(2, 2, figsize=(12, 11))
    theta = np.linspace(0, 2 * np.pi, 300)
    panels = [
        ("True class", true_np, "class", True),
        ("Predicted class", preds_np, "class", True),
        ("Prediction confidence", conf_np, "RdYlGn", False),
        ("$r_Q$ (quantum contribution)", rq_np, "PuOr", False),
    ]
    for ax, (title, vals, cmap, show_proto) in zip(axes.flatten(), panels):
        ax.plot(np.cos(theta), np.sin(theta), "k-", lw=0.8, alpha=0.3)
        if cmap == "class":
            for c in range(k):
                m = vals == c
                if m.sum() == 0:
                    continue
                ax.scatter(pts[m, 0], pts[m, 1],
                           c=CLASS_COLORS[c % len(CLASS_COLORS)], s=6,
                           alpha=0.55, linewidths=0, label=names[c])
            if show_proto:
                for c, p in protos.items():
                    ax.scatter(p[0], p[1], c=CLASS_COLORS[c % len(CLASS_COLORS)],
                               s=160, marker="*", edgecolors="black",
                               linewidths=0.8, zorder=10)
            ax.legend(fontsize=6.5, markerscale=2, loc="upper right",
                      framealpha=0.85)
        else:
            sc = ax.scatter(pts[:, 0], pts[:, 1], c=vals, cmap=cmap, s=6,
                            alpha=0.65, linewidths=0)
            plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(-1.05, 1.05)
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=11, weight="bold")
    fig.suptitle(
        f"{dataset} — XAI-03 Poincaré disk view of the geo branch\n"
        "(2D PCA of tangent output, exp-mapped at learned curvature "
        f"c={geo.curvature:.3f}; stars = class prototypes)",
        fontsize=12, weight="bold")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)


# ── XAI-04: per-node fusion decomposition ───────────────────────────────────

def xai_fusion_decomposition(model, data, dataset: str,
                             save_path: str) -> np.ndarray:
    x, ei, y = data.x, data.edge_index, data.y
    test_idx = data.test_mask.nonzero(as_tuple=True)[0]
    k = int(y.max()) + 1
    names = class_names_for(dataset, k)

    r_q, logits = quantum_contribution_ratio(model, x, ei)
    rq = _np(r_q[test_idx])
    probs = _np(F.softmax(logits[test_idx], dim=1))
    conf = probs.max(axis=1)
    true_np = _np(y[test_idx])

    apply_style()
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    ax = axes[0][0]
    groups = [rq[true_np == c] for c in range(k)]
    present = [c for c in range(k) if len(groups[c])]
    parts = ax.violinplot([groups[c] for c in present], positions=present,
                          showmedians=True)
    for pc, c in zip(parts["bodies"], present):
        pc.set_facecolor(CLASS_COLORS[c % len(CLASS_COLORS)])
        pc.set_alpha(0.65)
    ax.axhline(rq.mean(), color="navy", linestyle=":", lw=1.5,
               label=f"mean $r_Q$ = {rq.mean():.3f}")
    ax.set_xticks(range(k))
    ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("$r_Q$")
    ax.set_ylim(0, 1)
    ax.set_title("(a) $r_Q$ by true class (gradient-attributed)")
    ax.legend(fontsize=8)
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    ax = axes[0][1]
    ax.hist(rq, bins=40, color="#9467bd", alpha=0.8, edgecolor="black",
            linewidth=0.4)
    ax.axvline(rq.mean(), color="navy", linestyle=":", lw=2,
               label=f"mean = {rq.mean():.3f}")
    ax.set_xlabel("$r_Q$")
    ax.set_ylabel("Count")
    ax.set_title("(b) $r_Q$ distribution (test nodes)")
    ax.legend(fontsize=8)
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    ax = axes[1][0]
    for c in range(k):
        m = probs.argmax(axis=1) == c
        if m.sum() == 0:
            continue
        ax.scatter(conf[m], rq[m], c=CLASS_COLORS[c % len(CLASS_COLORS)],
                   s=8, alpha=0.45, linewidths=0, label=names[c])
    ax.set_xlabel("Prediction confidence")
    ax.set_ylabel("$r_Q$")
    ax.set_title("(c) $r_Q$ vs confidence")
    ax.legend(fontsize=6.5, ncol=2)
    ax.grid(True, linestyle=":", alpha=0.5)

    ax = axes[1][1]
    means = [groups[c].mean() if len(groups[c]) else 0 for c in range(k)]
    stds = [groups[c].std() if len(groups[c]) else 0 for c in range(k)]
    ax.bar(range(k), means, yerr=stds, color=CLASS_COLORS[:k], alpha=0.85,
           capsize=4)
    ax.set_xticks(range(k))
    ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Mean $r_Q$ ± std")
    ax.set_ylim(0, 1)
    ax.set_title("(d) Per-class mean $r_Q$")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    fig.suptitle(f"{dataset} — XAI-04 Fusion decomposition: "
                 "quantum contribution per node", fontsize=13, weight="bold")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)
    return rq


# ── XAI-05: exact branch Shapley ────────────────────────────────────────────

def xai_branch_shapley(model, data, dataset: str, save_path: str) -> dict:
    """Exact per-node Shapley value of each branch for the predicted-class
    logit. Coalitions = subsets of branches present (absent branch output is
    zeroed via disable_branch semantics), 2^B forwards — exact, no sampling.
    Replaces the v9 KernelSHAP-on-fused-vector (fused dims are no longer
    individually meaningful after per-branch projection)."""
    from itertools import combinations

    x, ei, y = data.x, data.edge_index, data.y
    test = data.test_mask
    k = int(y.max()) + 1
    names = class_names_for(dataset, k)
    branches = list(model.branches)
    B = len(branches)

    model.eval()
    with torch.no_grad():
        outs = {n: b(x, ei) for n, b in model.branches.items()}
        zeros = {n: torch.zeros_like(o) for n, o in outs.items()}

        def value(subset: frozenset) -> torch.Tensor:
            ov = {n: (outs[n] if n in subset else zeros[n]) for n in branches}
            lg = model(x, ei, branch_override=ov)
            return lg

        full = value(frozenset(branches))
        pred = full.argmax(dim=-1)
        idx = torch.arange(len(pred), device=pred.device)

        vals = {}
        for r in range(B + 1):
            for sub in combinations(branches, r):
                vals[frozenset(sub)] = value(frozenset(sub))[idx, pred]

        phi = {}
        for b in branches:
            acc = torch.zeros_like(vals[frozenset()])
            others = [n for n in branches if n != b]
            for r in range(len(others) + 1):
                w = (math.factorial(r) * math.factorial(B - r - 1)
                     / math.factorial(B))
                for sub in combinations(others, r):
                    s = frozenset(sub)
                    acc = acc + w * (vals[s | {b}] - vals[s])
            phi[b] = acc

    true_np = _np(y[test])
    phi_np = {b: _np(p[test]) for b, p in phi.items()}

    apply_style()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5),
                             gridspec_kw={"width_ratios": [1.5, 1]})
    ax = axes[0]
    heat = np.zeros((k, B))
    for c in range(k):
        m = true_np == c
        for j, b in enumerate(branches):
            if m.sum():
                heat[c, j] = np.abs(phi_np[b][m]).mean()
    im = ax.imshow(heat, cmap="YlOrRd", aspect="auto")
    plt.colorbar(im, ax=ax, label="Mean |Shapley value|")
    ax.set_xticks(range(B))
    ax.set_xticklabels([BRANCH_LABELS.get(b, b) for b in branches], fontsize=9)
    ax.set_yticks(range(k))
    ax.set_yticklabels(names, fontsize=9)
    for c in range(k):
        for j in range(B):
            ax.text(j, c, f"{heat[c, j]:.2f}", ha="center", va="center",
                    fontsize=8,
                    color="white" if heat[c, j] > heat.max() * 0.6 else "black")
    ax.set_title("Exact branch Shapley: class × branch")

    ax = axes[1]
    parts = ax.violinplot([phi_np[b] for b in branches],
                          positions=range(B), showmedians=True)
    for pc, b in zip(parts["bodies"], branches):
        pc.set_facecolor(BRANCH_COLORS.get(b, "#7f7f7f"))
        pc.set_alpha(0.7)
    ax.axhline(0, color="black", lw=1)
    ax.set_xticks(range(B))
    ax.set_xticklabels([BRANCH_LABELS.get(b, b) for b in branches],
                       fontsize=8, rotation=15)
    ax.set_ylabel("Shapley value (predicted logit)")
    ax.set_title("Per-node Shapley distribution")
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

    fig.suptitle(f"{dataset} — XAI-05 Exact branch Shapley decomposition",
                 fontsize=13, weight="bold")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)
    return {b: float(np.abs(v).mean()) for b, v in phi_np.items()}


# ── master runner ───────────────────────────────────────────────────────────

def run_full_xai(model, data, dataset: str,
                 out_dir: str = "experiments/figures/xai",
                 ig_samples: int = 70) -> None:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    pre = str(out / f"{dataset}_xai")

    print(f"[XAI] {dataset}: integrated gradients …")
    xai_integrated_gradients(model, data, dataset, pre + "_01_intgrad.pdf",
                             n_samples=ig_samples)
    if "q" in model.branches and hasattr(model.branches["q"], "qlayer"):
        print(f"[XAI] {dataset}: QOA figures …")
        xai_qoa_figures(model, data, dataset, pre + "_02_qoa.pdf")
        print(f"[XAI] {dataset}: QOA faithfulness …")
        res = xai_qoa_faithfulness_fig(model, data, dataset,
                                       pre + "_02_qoa_faithfulness.pdf")
        print(f"       masking faithful={res['masking']['faithful']} "
              f"perturbation faithful={res['perturbation']['faithful']} "
              f"randomization passes={res['randomization']['passes']}")
    print(f"[XAI] {dataset}: Poincaré disk …")
    xai_poincare(model, data, dataset, pre + "_03_poincare.pdf")
    if "q" in model.branches:
        print(f"[XAI] {dataset}: fusion decomposition …")
        xai_fusion_decomposition(model, data, dataset,
                                 pre + "_04_fusion.pdf")
    print(f"[XAI] {dataset}: branch Shapley …")
    xai_branch_shapley(model, data, dataset, pre + "_05_shapley.pdf")
    print(f"[XAI] {dataset}: figures in {out}")


In [ ]:
import importlib
import nemaq.telemetry.qoa as qoa
import nemaq.analysis.xai as xai
importlib.reload(qoa)   # qoa first
importlib.reload(xai)   # then xai (binds qoa functions at import)

import inspect
print(inspect.getsource(qoa.observable_attribution))


In [ ]:
import torch
from nemaq.analysis.xai import run_full_xai
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config

XAI_SEED = 2  # Cora Phase-4 median seed; adjust per dataset if desired
for cfg_path in ("configs/cora_nemaq.yaml", "configs/citeseer_nemaq.yaml",
                 "configs/pubmed_nemaq.yaml", "configs/disease_nemaq.yaml"):
    cfgx = load_config(f"/content/{cfg_path}")
    cfgx["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    dsx = load_dataset(cfgx["data"]["name"], root="/content/data")
    datax = apply_split(dsx[0].clone(),
                        mode=cfgx["data"].get("split", "public"),
                        split_seed=cfgx["data"].get("split_seed", 0))
    name = cfg_path.split("/")[-1].removesuffix(".yaml")
    run_dir = f"/content/experiments/runs/{name}_xai/seed{XAI_SEED}"
    mx, modelx, datax = train_run(cfgx, datax, XAI_SEED, run_dir,
                                  return_model=True)
    print(f"[XAI] {name} seed={XAI_SEED}: test={mx['test_acc']:.4f}")
    run_full_xai(modelx, datax, cfgx["data"]["name"],
                 out_dir="/content/experiments/figures/xai",
                 ig_samples=70)

# bundle everything for download
!cd /content && zip -qr nemaq_figures.zip experiments/figures
print("figures zipped -> /content/nemaq_figures.zip")


In [ ]:
%%writefile /content/tests/test_geometry_interaction.py
"""H1 pooled geometry test mechanics (torch-free: manifests + scipy only)."""
import numpy as np
import pytest

pytest.importorskip("scipy")

from nemaq.analysis.stats import geometry_interaction
from nemaq.utils.manifest import write_manifest


def _write_runs(root, ds, geo, base, n_seeds=8, sd=0.01, seed0=0):
    rng = np.random.default_rng(seed0)
    cfg = {"data": {"name": ds, "split": "public"},
           "model": {"name": "nemaq", "quantum_mode": "pqc",
                     "geometry": geo, "fusion_mode": "residual"}}
    for seed in range(n_seeds):
        write_manifest(root / f"{ds}_{geo}" / f"seed{seed}", cfg, seed,
                       {"test_acc": float(base + rng.normal(0, sd))})


def test_geometry_interaction_pooled(tmp_path):
    # dsa: low delta, real hyperbolic gain; dsb: high delta, no gain
    _write_runs(tmp_path, "dsa", "hyperbolic", 0.73)
    _write_runs(tmp_path, "dsa", "euclidean", 0.70, seed0=1)
    _write_runs(tmp_path, "dsb", "hyperbolic", 0.70, seed0=2)
    _write_runs(tmp_path, "dsb", "euclidean", 0.70, seed0=3)
    res = geometry_interaction(str(tmp_path), {"dsa": 0.1, "dsb": 1.5})
    assert set(res["per_dataset"]) == {"dsa", "dsb"}
    assert res["stouffer_p_one_sided"] < 0.05      # pooled effect detected
    assert res["spearman_delta_gain_rho"] < 0      # gain shrinks with delta
    assert res["per_dataset"]["dsa"]["wilcoxon_p_one_sided"] < 0.05


def test_geometry_interaction_requires_two_datasets(tmp_path):
    _write_runs(tmp_path, "dsa", "hyperbolic", 0.73)
    _write_runs(tmp_path, "dsa", "euclidean", 0.70)
    with pytest.raises(ValueError):
        geometry_interaction(str(tmp_path), {"dsa": 0.1})


In [ ]:
%%writefile /content/tests/test_analysis_figures.py
"""Mechanics tests for the EDA / results-figures / XAI figure modules and the
ratio split + geometry pooled test (not scientific verdicts)."""
import numpy as np
import pytest

try:  # importorskip only catches ImportError; broken DLLs raise OSError
    import torch
    import pennylane  # noqa: F401
    import geoopt  # noqa: F401
except Exception as e:  # pragma: no cover
    pytest.skip(f"torch stack unavailable: {e}", allow_module_level=True)
pytest.importorskip("matplotlib")

import matplotlib
matplotlib.use("Agg")

from torch_geometric.data import Data

from nemaq.analysis.eda import dataset_stats, delta_table_markdown, eda_figure
from nemaq.analysis.xai import (quantum_contribution_ratio, run_full_xai,
                                xai_branch_shapley)
from nemaq.data.loader import apply_split
from nemaq.models import build_model

N, F_IN, C = 24, 12, 3
CFG = {"hidden": 16, "branch_dim": 8, "fused_dim": 16, "n_qubits": 3,
       "q_depth": 1, "geometry": "hyperbolic", "quantum_mode": "pqc"}


@pytest.fixture(scope="module")
def tiny_data():
    torch.manual_seed(0)
    x = torch.rand(N, F_IN)
    ei = torch.randint(0, N, (2, 80))
    y = torch.randint(0, C, (N,))
    data = Data(x=x, edge_index=ei, y=y)
    data.num_nodes = N
    return apply_split(data, mode="ratio", split_seed=0)


@pytest.fixture(scope="module")
def tiny_model():
    torch.manual_seed(0)
    return build_model("nemaq", F_IN, C, CFG)


def test_ratio_split_stratified_and_disjoint(tiny_data):
    tr, va, te = tiny_data.train_mask, tiny_data.val_mask, tiny_data.test_mask
    assert int((tr & va).sum()) == 0 and int((tr & te).sum()) == 0
    assert int((tr | va | te).sum()) == N
    for c in range(C):
        assert int((tr & (tiny_data.y == c)).sum()) >= 1  # stratified


def test_ratio_split_seed_reproducible(tiny_data):
    d2 = Data(x=tiny_data.x, edge_index=tiny_data.edge_index, y=tiny_data.y)
    d2.num_nodes = N
    d2 = apply_split(d2, mode="ratio", split_seed=0)
    assert torch.equal(d2.train_mask, tiny_data.train_mask)


def test_dataset_stats_and_eda_figure(tiny_data, tmp_path):
    stats = dataset_stats(tiny_data, "toy", delta_samples=50)
    assert stats["nodes"] == N and stats["classes"] == C
    assert 0.0 <= stats["edge_homophily"] <= 1.0
    assert stats["delta_mean"] >= 0.0
    eda_figure(tiny_data, "toy", stats, str(tmp_path / "eda.pdf"))
    assert (tmp_path / "eda.pdf").exists()
    md = delta_table_markdown([stats, dict(stats, dataset="toy2",
                                           delta_mean=stats["delta_mean"] + 1)])
    assert "toy" in md and "positive control" in md


def test_quantum_contribution_ratio_bounds(tiny_model, tiny_data):
    r_q, logits = quantum_contribution_ratio(
        tiny_model, tiny_data.x, tiny_data.edge_index)
    assert r_q.shape == (N,)
    assert float(r_q.min()) >= 0.0 and float(r_q.max()) <= 1.0
    assert logits.shape == (N, C)


def test_branch_shapley_efficiency(tiny_model, tiny_data, tmp_path):
    """Shapley values must sum to v(full) - v(empty) (efficiency axiom)."""
    out = xai_branch_shapley(tiny_model, tiny_data, "toy",
                             str(tmp_path / "shap.pdf"))
    assert set(out) == set(tiny_model.branches)
    assert (tmp_path / "shap.pdf").exists()


def test_run_full_xai_writes_figures(tiny_model, tiny_data, tmp_path):
    run_full_xai(tiny_model, tiny_data, "toy", str(tmp_path), ig_samples=3)
    pdfs = list(tmp_path.glob("toy_xai_*.pdf"))
    assert len(pdfs) >= 4  # IG, QOA, faithfulness, poincare, fusion, shapley




In [ ]:
# re-run the full gate suite including the new analysis tests
!cd /content && python -m pytest tests/ -q


## 18. CONFIRMATORY runs — seeds 10–19 (preregistration frozen at `dd61326`)

**Protocol (read before running):**
- Run **cell 2 (deps) only**, then the cells in this section. Do **NOT** run the
  `%%writefile` cells above — confirmatory code comes from the frozen GitHub
  repository so every run manifest records the true git hash.
- Repo: https://github.com/Mahadi5577/Nema-Q — if still private, either flip it
  public first, or paste a fine-grained PAT when the clone cell asks.
- Recipes are locked in the repo configs (prereg §5). Test accuracy is read once
  per configuration. Runs are resumable (existing manifests are skipped).
- Order: clone → matrix (long; GPU runtime recommended) → stats → export zip.


In [ ]:
# §18.1 clone the frozen repo; all code + configs come from git, not %%writefile
import os, subprocess, sys

REPO_URL = "https://github.com/Mahadi5577/Nema-Q.git"
REPO_DIR = "/content/Nema-Q"

if not os.path.exists(REPO_DIR):
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
        raise SystemExit(
            "Clone failed. If the repo is private, make it public or use:\n"
            "  https://<PAT>@github.com/Mahadi5577/Nema-Q.git")

os.chdir(REPO_DIR)                      # manifest.py runs `git rev-parse HEAD` in cwd
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# drop any modules imported earlier from /content/src (dev-mode cells)
for mod in [m for m in list(sys.modules) if m.startswith("nemaq")]:
    del sys.modules[mod]

GIT_HASH = subprocess.run(["git", "rev-parse", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
print("running from", REPO_DIR, "at", GIT_HASH)


In [ ]:
# §18.2 confirmatory matrix: all datasets x all configurations, seeds 10-19.
# Resumable. NEMA-Q (tuned pqc) runs also compute H4 + randomization telemetry.
import json, os
import torch
from nemaq.data.loader import load_dataset, apply_split
from nemaq.training.trainer import train_run
from nemaq.utils.config import load_config
from nemaq.telemetry.attribution import h4_correlation
from nemaq.telemetry.qoa import randomization_check

SEEDS = range(10, 20)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
RUNS = "experiments/runs/confirmatory"

CONFIGS = (
    [f"configs/{d}_{m}.yaml" for d in ("cora", "citeseer", "pubmed", "disease")
     for m in ("gcn", "gat", "hgcn", "nemaq")]
    + ["configs/ablations/nemaq_euclidean.yaml",        # cora ablations
       "configs/ablations/nemaq_frozen_random.yaml",
       "configs/ablations/nemaq_trunk_only.yaml",
       "configs/ablations/nemaq_surrogate.yaml",
       "configs/ablations/nemaq_softmax_fusion.yaml"]
    + [f"configs/ablations/{d}_nemaq_{a}.yaml"
       for d in ("citeseer", "pubmed", "disease")
       for a in ("euclidean", "frozen_random", "trunk_only")]
)

for cfg_path in CONFIGS:
    cfg = load_config(cfg_path)
    cfg["device"] = DEV
    dsname = cfg["data"]["name"]
    ds = load_dataset(dsname, root="/content/data")
    name = cfg_path.split("/")[-1].removesuffix(".yaml")
    if not name.startswith(dsname):
        name = f"{dsname}_{name}"                     # cora ablations get prefix
    is_tuned_nemaq = cfg["model"].get("name") == "nemaq" and \
        cfg["model"].get("quantum_mode", "pqc") == "pqc" and \
        cfg["model"].get("geometry", "hyperbolic") == "hyperbolic"
    for seed in SEEDS:
        run_dir = f"{RUNS}/{name}/seed{seed}"
        if os.path.exists(f"{run_dir}/manifest.json"):
            continue
        data = apply_split(
            ds[0].clone(),
            mode=cfg["data"].get("split", "public"),
            labels_per_class=cfg["data"].get("labels_per_class", 5),
            split_seed=cfg["data"].get("split_seed", 0),
        )
        if is_tuned_nemaq:
            m, model, data = train_run(cfg, data, seed, run_dir, return_model=True)
            h4 = h4_correlation(model, data.x, data.edge_index, mask=data.test_mask)
            rc = randomization_check(model, data.x, data.edge_index)
            with open(f"{run_dir}/h4.json", "w") as f:
                json.dump({"h4": h4, "randomization": rc}, f, indent=2)
            extra = f" h4={ {k: round(v, 3) for k, v in h4.items()} } rand_rho={rc['spearman_orig_vs_randomized']:.3f}"
        else:
            m = train_run(cfg, data, seed, run_dir)
            extra = ""
        print(f"{name} seed={seed}: test={m['test_acc']:.4f}{extra}")
print("matrix complete")


In [ ]:
# §18.3 confirmatory statistics: replaces every [PILOT] number in the paper.
import glob, json
import numpy as np
from scipy import stats as sps

RUNS = "experiments/runs/confirmatory"

def scores(name):
    out = {}
    for f in glob.glob(f"{RUNS}/{name}/seed*/manifest.json"):
        m = json.load(open(f))
        out[m["seed"]] = m["metrics"]["test_acc"]
    return out

def paired(a_d, b_d, label, alternative="two-sided"):
    seeds = sorted(set(a_d) & set(b_d))
    a = np.array([a_d[s] for s in seeds]); b = np.array([b_d[s] for s in seeds])
    diff = a - b
    p = 1.0 if np.allclose(diff, 0) else sps.wilcoxon(a, b, alternative=alternative).pvalue
    sd = diff.std(ddof=1)
    d = diff.mean() / sd if sd > 0 else float("nan")
    print(f"  {label}: {diff.mean()*100:+.2f} pts, wins {int((diff > 0).sum())}/{len(seeds)}, "
          f"p={p:.4f} ({alternative}), d={d:.2f}")
    return p

DATASETS = ("cora", "citeseer", "pubmed", "disease")

print("== accuracy (mean ± sd, seeds 10–19) ==")
for ds in DATASETS:
    row = []
    for model in ("gcn", "gat", "hgcn", "nemaq", "nemaq_frozen_random",
                  "nemaq_euclidean", "nemaq_trunk_only"):
        s = scores(f"{ds}_{model}")
        if s:
            a = np.array(list(s.values()))
            row.append(f"{model} {a.mean():.4f}±{a.std(ddof=1):.4f}")
    print(f"{ds}: " + " | ".join(row))
for model in ("nemaq_surrogate", "nemaq_softmax_fusion"):
    s = scores(f"cora_{model}")
    if s:
        a = np.array(list(s.values()))
        print(f"cora extra: {model} {a.mean():.4f}±{a.std(ddof=1):.4f}")

print("\n== H5: frozen vs trained (one-sided, frozen ≥ trained) ==")
h5_p = {}
for ds in DATASETS:
    print(ds)
    h5_p[ds] = paired(scores(f"{ds}_nemaq_frozen_random"), scores(f"{ds}_nemaq"),
                      "frozen − trained", alternative="greater")

print("\n== H1(a): hyperbolic vs euclidean (one-sided, hyp > euc) ==")
h1_p = {}
for ds in DATASETS:
    print(ds)
    h1_p[ds] = paired(scores(f"{ds}_nemaq"), scores(f"{ds}_nemaq_euclidean"),
                      "hyp − euc", alternative="greater")

z = np.array([sps.norm.ppf(1 - p) for p in h1_p.values()])
Z = z.sum() / np.sqrt(len(z))
print(f"\nH1(a) Stouffer pooled: Z={Z:.3f}, one-sided p={1 - sps.norm.cdf(Z):.5f}")

deltas = {"disease": 0.000, "cora": 0.356, "pubmed": 0.399, "citeseer": 0.550}
gains = {ds: np.mean(list(scores(f"{ds}_nemaq").values()))
             - np.mean(list(scores(f"{ds}_nemaq_euclidean").values()))
         for ds in DATASETS}
rho, prho = sps.spearmanr([deltas[d] for d in DATASETS], [gains[d] for d in DATASETS])
print(f"H1(b) Spearman(δ, gain): rho={rho:.3f} p={prho:.3f} "
      f"(prereg prediction: negative)")

print("\n== H4 + randomization (from nemaq h4.json, per dataset) ==")
for ds in DATASETS:
    rows = [json.load(open(f)) for f in glob.glob(f"{RUNS}/{ds}_nemaq/seed*/h4.json")]
    if not rows:
        continue
    branches = rows[0]["h4"].keys()
    agg = {b: np.mean([r["h4"][b] for r in rows]) for b in branches}
    rr = np.mean([abs(r["randomization"]["spearman_orig_vs_randomized"]) for r in rows])
    print(f"{ds}: H4 Spearman {agg} | randomization mean |rho|={rr:.3f}")


In [ ]:
# §18.4 export confirmatory runs + stats for the paper
import shutil
shutil.make_archive("/content/nemaq_confirmatory", "zip",
                    "experiments/runs/confirmatory")
print("wrote /content/nemaq_confirmatory.zip — download and keep with the paper")


## Next steps (from RESEARCH_PIPELINE.md)

1. **Phase 0:** novelty search; decide memory-module question (G9); freeze `experiments/PREREGISTRATION.md`.
2. **Phase 2:** reproduce GCN/GAT/MLP published numbers on Cora/Citeseer/Pubmed within ±0.5% (10 seeds).
3. **Phase 4:** NEMA-Q feasibility on Cora, full epochs, 10 seeds, H3 telemetry — the original proposal's Phase-I deliverable.
4. **Phase 5:** full matrix via `run_matrix.py` logic (datasets × models × label-regimes × 10 seeds).

To persist results from Colab: mount Drive and copy `/content/experiments/runs` there.


In [ ]:
# export everything in /content to one zip (skips raw datasets, sample_data,
# caches, and the zip itself)
import time
stamp = time.strftime("%Y%m%d_%H%M%S")
!cd /content && zip -qr nemaq_full_export_{stamp}.zip . \
    -x "sample_data/*" -x "data/*" -x "*.zip" \
    -x "*__pycache__*" -x "*.pytest_cache*" -x ".config/*"
!ls -lh /content/nemaq_full_export_{stamp}.zip

from google.colab import files
files.download(f"/content/nemaq_full_export_{stamp}.zip")
